In [ ]:
import json
import os
import re
from pathlib import Path
import pandas as pd
import pandas_dedupe
import duckdb
import mysql.connector
import subprocess
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

class CBSExtraction:

    def __init__(self, path_to_databases,path_to_json):
        self.folder_path = path_to_databases
        self.json_path = path_to_json
    
    def get_connection(self):
        connection = mysql.connector.connect(user="root", password="root", host="localhost", port="3306")
        return connection


    def extracting_data(self,sql,connection):
        connection = self.get_connection()
        df = pd.read_sql(sql,connection)
        connection.close()
        return df

    def get_facility_databases(self,folder_path): 
        facility_databases = []
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.endswith(".sql"):
                    file_path = os.path.join(root, file)
                    facility_databases.append(file_path)
        return facility_databases

    def restore_database(self,backup_file,time_):
        backup_file = backup_file.replace("\\", "/")
        # FIRST CHECK FOR EXISTING DATABASES AND DROP THEM
        connection = self.get_connection()

        results = pd.read_sql('SELECT schema_name FROM information_schema.schemata;', con=connection)
        schemas = list(results["schema_name"])
        # Create a cursor object
        cursor = connection.cursor()
        # Execute a query
        cursor.execute(f'SET foreign_key_checks = 0')
        for schema in schemas:
            if schema in ['client', 'consultation', 'deduplication', 'facility', 'mrs', 'provider', 'report', 'terminology', 'zimepms']:
                results = cursor.execute(f'DROP DATABASE {schema}')
                print("DROPPED [{0}]".format(schema))
        cursor.close()
        restore_command = f'mysql -u root -proot -f < {backup_file}'
        try:
            print(">>> Restoring database")
            subprocess.run(restore_command, shell=True, check=True)
            print(f'DATABASE RESTORE [SUCCESSFUL>>>]')
        except Exception as e:
            with open('./logs.txt', 'a') as f:
                f.write(time_ + f'{backup_file}\n')

    @staticmethod
    def get_facility_identifiers(metadata):
        """Returns the siteId from the metadata string."""
        siteId = re.search(r"<string>siteId</string><string>(.*?)</string>", metadata)
        version = re.search(r"<string>version</string><string>(.*?)</string>", metadata)
        last_time_stamp = re.search(r"<string>time</string><string>(.*?)</string>", metadata)
        if siteId:
            return siteId.group(1) , version.group(1) , last_time_stamp.group(1)
        else:
            return "","",""
    
    @staticmethod
    def get_first_time_stamp(metadata):
        """Returns the siteId from the metadata string."""
        first_time_stamp = re.search(r"<string>time</string><string>(.*?)</string>", metadata)
        if first_time_stamp:
            return first_time_stamp.group(1) 
        else:
            return ""
        
    @staticmethod
    def get_size_of_folder(filename):
        folder_size = round(os.path.getsize(filename)/(1000*1024),2)
        return folder_size
    

    def split_string(self,string):
        # Match all commas outside of quotes
        pattern = r",(?=(?:[^']*'[^']*')*[^']*$)"
        return re.split(pattern, string)
    
    def save_json(self,dictionary,filename):
        with open(self.json_path + filename + ".json", "w") as fp:
            json.dump(dictionary,fp)

    def read_json(self,facility_id):
        with open(self.json_path + facility_id + ".json", "r") as fp:
            data = json.load(fp)
        facility_id = list(data.keys())[0]
        return data , facility_id
    
    
    def construct_table_data(self,filename):
        with open(os.path.join(self.folder_path, filename), "r", encoding="ISO-8859-1") as f:
            content = f.read()
        # content = content.replace("Unprotected sex (sex without a condom)", "Unprotected sex without a condom"
        content = re.sub(r"\)'", "", content)
        subText = "',',"
        content = content.replace(subText,"','")
        latest_site_id = []
        latest_timestamp = []
        first_timestamp = []
        version = []
        
        domain_event = ""
        if "/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;" in content:
            domain_event = content.split("/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;")
        else:
            latest_site_id = "corrupt"
            fact_tables = {}
            row = ""
            first_timestamp =  ""
            version = ""
            return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables

        if "/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */" in domain_event[1]:
            domain_event = domain_event[1].split("/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */")
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[0])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[0])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[0])
        else:
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[1])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[1])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[1])
        if facility_search:
            latest_site_id = facility_search[-1]
            latest_timestamp = time_serach[-1]
            first_timestamp = time_serach[0]
            if len(version)==0:
                version = ""
            else:
                version = version[-1]
        db_tables = {}
        fact_tables = {}
        database_statements = re.split(r'(?<=\n)CREATE DATABASE', content)
        for statement in database_statements[1:]:
            table_names = []
            database_name = re.findall(r'`(\w+)`', statement)[0]

            if database_name not in ("client","consultation","report"):
                 continue
            # Use regular expressions to extract the table names
            pattern = re.compile(r"CREATE TABLE `(.+?)`", re.DOTALL)
            matches = pattern.findall(statement)
            
            # Loop through the matches and append the table names to the list
            for match in matches:
                table_names.append(match)

            
            # Create a dictionary of dataframes for the tables
            tables_dict = {}
            for table_name in table_names:

                # if table_name not in table_list:
                #     continue
                table_str = re.search(rf'CREATE TABLE `{table_name}`.+?ENGINE=InnoDB DEFAULT CHARSET', statement, re.DOTALL)
                if table_str is None:
                    continue
                table_str  = table_str.group(0).strip()
                columns = re.findall(r'`(\w+)` (\w+)', table_str)
                column_names = [column[0] for column in columns if not column[0].startswith("fk")]
                data = re.findall(rf'INSERT INTO `{table_name}` VALUES \(.+?\);', statement, re.DOTALL)
                rows = {}
                if data:
                    values_list = []
                    for row in data: 

                        row = re.findall(r'\((.*?)\)', row)
                        # row = row.split('(',1)[1]
                        
                        values2 = [ re.split(r',(?=")', s) for s in row]
            
                        for item in values2:
                            item = item[0].replace('NULL', "'NULL'")
                            rec = [i.replace("'","").lstrip() for i in item.split("',")]
                            values_list.append(rec)
                     
                    for column in column_names:
                        coulum_data = []
                        column_index = column_names.index(column)
                        for record in values_list:
                            if column_index < len(record):
                                coulum_data.append(record[column_index])
                            else:
                                coulum_data.append("")  
                        rows[column] = coulum_data
                    table_df = pd.DataFrame(rows)
                    table_df = table_df.to_dict(orient="records")
                    tables_dict[table_name] = table_df
                # else:
                #     table_df = pd.DataFrame(columns=column_names)
                #     tables_dict[table_name] = table_df
                # Add the database and its tables to the dictionary
            db_tables[database_name] = tables_dict

            if len(latest_site_id)==0:
                latest_site_id = ""
                fact_tables = {}
                row = ""
                first_timestamp =  ""
                version = ""
                return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables
            
        fact_tables[latest_site_id] = db_tables

        return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables
    
    def get_database_size(self,filename):
        size = round(Path(self.folder_path + filename).stat().st_size /(1000*1024),2)
        return size
    
    @staticmethod
    def log(processing_time,province="",district="",facility="",site_code="",filename="",first_date="",last_date="",
            db_size="",version="",comment="",ehr_type = ""): 
         df = pd.DataFrame({"Time File Processed":[processing_time],
                            "province": [province],
                            "district": [district],
                            "facility": [facility],
                            "Site Code":[site_code],
                            "File Name":[filename],
                            "Date site first used EHR": [first_date],
                            "Date site last used EHR": [last_date],
                            "Database Size": [db_size],
                            "Version":[version],
                            "Comments":[comment],
                            "EHR Type":[ehr_type]
                            })
         df.to_csv('log.csv', mode='a', index = False, header=None)
    
    @staticmethod
    def dbTableList():
        # List of tables to use for the program / this reduce time for table reconstruction by focusing only on tables to be used
        table_list = ["hts","hts_screening","person_investigation","patient","cbs",
                      "sexual_history","sexual_history_question","art","art_visit",
                      "art_current_status","tb","art_who_stage","person",
                      "art_appointment","laboratory_investigation_test","laboratory_request_order",
                      "art_transfer_out","patient_tb_screening","patient_client_profile"]
        return table_list
    
    @staticmethod
    def get_mapping_file():
        mapping_file = pd.read_csv("mapping_file.csv")
        mapping_file['Facility ID'] = mapping_file['Facility ID'].str.strip()
        return mapping_file
    
    @staticmethod
    def get_facility_details(mapping_file,facility_id):
        facility_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["Facility"].values
        if facility_name.size > 0:
            facility_name = facility_name[0]
            district_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["District"].values
            if district_name.size > 0:
                district_name = district_name[0]
            else:
                district_name = ""
            province_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["Province"].values
            if province_name.size > 0:
                province_name = province_name[0]
            else:
                province_name = ""
        else:
            facility_name = ""
            district_name = ""
            province_name = ""
        return facility_name,district_name,province_name
    
    @staticmethod
    def deduplication(demographics):
        clusters = pandas_dedupe.dedupe_dataframe(demographics,['FirstName','LastName', 'Birthdate', 'Sex', "Street", "City","Education Id"],n_cores = 0)
        return clusters

    @staticmethod
    def cbs_unique_id(record,seq_number):
        if record["person_id"] == "":
            cbs_id = "No Demographics"
        elif record["cluster id"] in cluster_id_dictionary.keys():
            cbs_id = cluster_id_dictionary.get(record["cluster id"])
        else:
            
            first_letter_first_name = str(record["firstname"])[0]
            last_letter_first_name = str(record["firstname"])[-1]
            first_letter_last_name = str(record["Lastname"])[0]
            last_letter_last_name = str(record["Lastname"])[-1]
            first_letter_of_sex = str(record["sex"])[0]
            birthdate = str(record["birthdate"]).replace("/","")
            birthdate = str(record["birthdate"]).replace("-","")
            sequential_number = str(seq_number).rjust(6,"0")
            cbs_id = first_letter_first_name + last_letter_first_name + "-" + first_letter_last_name + last_letter_last_name + "-" + first_letter_of_sex + "-" + birthdate + "-" + sequential_number
            cluster_id_dictionary[ record["cluster id"]] = cbs_id
        return cbs_id

    @staticmethod
    def concat_dataframes(list_of_dataframes):
        df = pd.concat(list_of_dataframes, ignore_index=True, sort=False)
        return df
    
    @staticmethod
    def remove_people_without_positive_result(df,df_positives):
        df = df[df["CBS ID"].isin(list(df_positives["CBS ID"]))]
        return df
    
    @staticmethod
    def get_last_negative_date(df_negatives,df_positives):
        list_all = []
        cbs_ids= list(df_negatives["CBS ID"])
        # Pick the first positive for each cbs id
        df_positives.sort_values(by =['CBS ID','Date of HIV Test'],ascending = True, inplace= True)
        df_positives.drop_duplicates('CBS ID',keep = 'first',inplace= True)
        # Create mapping dictionary
        mapping_dict = dict(zip(df_positives['CBS ID'], df_positives['Date of HIV Test']))
        df_negatives['Date of HIV Test'] = df_negatives["CBS ID"].map(mapping_dict)
        for cbs_id in cbs_ids:
            df_cbs = df_negatives[df_negatives["CBS ID"] == cbs_id]
            sorted_df = df_cbs.sort_values('Last HIV negative date', ascending=False)
            list_negative = [lnd for lnd in sorted_df.iterrows() 
                             if lnd[1]["Last HIV negative date"] + pd.Timedelta(days=1) < lnd[1]["Date of HIV Test"]]
            if len(list_negative)> 0:
                list_all.append(list_negative[0][1])
        list_all_df = pd.DataFrame(list_all)
        list_all_df = list_all_df.drop('Date of HIV Test', axis=1)
        list_all_df = list_all_df[~list_all_df["Event_date"].isna()]
        return list_all_df
    
  

if __name__ == '__main__':

    processing_time = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    extraction  = CBSExtraction("./", "./Jason Files/")
    connection = extraction.get_connection()
    facilities = extraction.get_facility_databases(r"C:\Users\Simba\Desktop\Harare")
    mapping_file = extraction.get_mapping_file()

    #----------------Lists for accumulative data from facilities..........................................
    global_df_demos = pd.DataFrame()
    global_df_hts_positives = pd.DataFrame()
    global_df_hts_negative = pd.DataFrame()
    global_df_cbs = pd.DataFrame()
    global_df_recency = pd.DataFrame()
    global_df_art = pd.DataFrame()
    global_df_art_visit = pd.DataFrame()
    global_df_art_current_status = pd.DataFrame()
    global_df_viral_load = pd.DataFrame()
    global_df_cd4 = pd.DataFrame()
    global_df_tb = pd.DataFrame()
    global_df_tb_screening = pd.DataFrame()
    global_df_transfer_out =  pd.DataFrame()
    global_df_art_who_stage = pd.DataFrame()

    print(">>> Number of databases ", len(facilities))
    

    if Path(extraction.folder_path+ "log.csv").exists():
        os.remove("log.csv")
    if Path(extraction.folder_path+ "demos.csv").exists():
        os.remove("demos.csv")
    
    
    for facility in facilities:
        print(">>>", facility)

        # Restore database
        extraction.restore_database(facility,processing_time)

        # -------------Section1 : Constructing dataframes from database tables -------------------------------------
        print("   >>> Constructing database tables")

        
        # ----- get facility id,version and last time stamp
        df_facility = extraction.extracting_data("""
                            SELECT meta_data FROM mrs.domain_event_entry
                            order by time_stamp desc
                            limit 1
                        """, connection)
        facility_id,version,last_timestamp = extraction.get_facility_identifiers(df_facility.iloc[0][0].decode())

        facility_name , district_name, province_name =  extraction.get_facility_details(mapping_file,facility_id)

        # get facility first time stamp
        first_time_stamp = extraction.extracting_data("""
                                                            SELECT meta_data FROM mrs.domain_event_entry
                                                            order by time_stamp asc
                                                            limit 1
                                                          """, connection)
        get_first_time_stamp = extraction.get_first_time_stamp(first_time_stamp.iloc[0][0].decode())
   

       # Patient ----------------
        patient = extraction.extracting_data("""
                                                SELECT * FROM consultation.patient
                                                          """, connection)
 
        # HTS  ------------
        hts = extraction.extracting_data("""
                                                SELECT * FROM consultation.hts
                                                          """, connection)

        # HTS Screeniing------------
        hts_screening = extraction.extracting_data("""
                                                SELECT * FROM consultation.hts_screening
                                                          """, connection)

        # Person Investigation 
        person_investigation = extraction.extracting_data("""
                                                SELECT * FROM consultation.person_investigation
                                                          """, connection)
  
        # cbs
        cbs = extraction.extracting_data("""
                                                SELECT * FROM consultation.cbs
                                                          """, connection)
        

        # sexual_history ----------------------
        sexual_history = extraction.extracting_data("""
                                                SELECT * FROM consultation.sexual_history
                                                          """, connection)

        # sexual_history_question ----------------------
        sexual_history_question = extraction.extracting_data("""
                                                SELECT * FROM consultation.sexual_history_question
                                                          """, connection)

        # patient_tb_screening ----------------------
        tb_screening = extraction.extracting_data("""
                                                SELECT * FROM consultation.patient_tb_screening
                                                          """, connection)

        # patient_client_profile ----------------------
        patient_client_profile = extraction.extracting_data("""
                                                SELECT * FROM consultation.patient_client_profile
                                                          """, connection)
        
        
         # art_transfer_out ----------------------
        art_transfer_out = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_transfer_out
                                                          """, connection)

         # tb ----------------------
        tb = extraction.extracting_data("""
                                                SELECT * FROM consultation.tb
                                                          """, connection)

        # laboratory_investigation_test ----------------------
        laboratory_investigation_test = extraction.extracting_data("""
                                                SELECT * FROM consultation.laboratory_investigation_test
                                                          """, connection)

        # laboratory_request_order ----------------------
        lab_request = extraction.extracting_data("""
                                                SELECT * FROM consultation.laboratory_request_order
                                                          """, connection)

        # art_visit ----------------------
        art_visit = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_visit
                                                          """, connection)

        # art_who_stage ----------------------
        art_who_stage = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_who_stage
                                                          """, connection)
        
        # art_current_status ----------------------
        art_current_status = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_current_status
                                                          """, connection)
        
        # art_appointment ----------------------
        art_appointment = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_appointment
                                                          """, connection)
        
        # art ----------------------
        art = extraction.extracting_data("""
                                                SELECT * FROM consultation.art
                                                          """, connection)

        # person_diagnosis ----------------------
        person_diagnosis = extraction.extracting_data("""
                                                SELECT * FROM consultation.person_diagnosis
                                                          """, connection)

        # person ----------------------
        demographics = extraction.extracting_data("""
                                                SELECT * FROM client.person
                                                          """, connection)

    
        demographics["birthdate"] = pd.to_datetime(demographics["birthdate"],infer_datetime_format=True,errors='coerce')
        person_diagnosis["date_created"] = pd.to_datetime(person_diagnosis["date_created"],infer_datetime_format=True,errors='coerce')
        patient["time"] = pd.to_datetime(patient["time"],infer_datetime_format=True,errors='coerce')
        hts["time"] = pd.to_datetime(hts["time"],infer_datetime_format=True,errors='coerce')


        # -------------Section2 : Extract data using SQL into ----------------------------------------------------------------------------
        
        print("   >>> Extracting data")

        #---------------a. new HIV positives
        df_hts = duckdb.query("""
                            WITH ranked_messages AS (
                            select 
                            p.person_investigation_id,
                            pt.time as Event_date,
                            p.person_id,
                            h.hts_number as 'HTS Number',
                            reason_for_not_initiating_art,
                            pt.time as 'Date of HIV Test',
                            h.entry_point as 'Entry Point',
                            h.purpose as 'Reason for HIV Test',
                            h.reason_for_not_performing_test as 'Reason for not doing recency test',
                            h.entry_point_id as 'Entry Point Id', h.hts_type as 'Test Method', 
                            h.client_already_positive as 'Already positive',
                            h.client_already_on_art as 'Already on art', p.result as HIV_Result,
                            h.pregnant as 'Pregnant during HIV Test',
                            ROW_NUMBER() OVER (PARTITION BY p.person_id ORDER BY pt.time ASC) as rnt
                            FROM hts h  
                            left join patient pt
                            on  h.patient_id = pt.patient_id
                            left join person_investigation p
                            on h.laboratory_investigation_id = p.person_investigation_id
                            where  p.test = 'HIV' and HIV_Result = 'POSITIVE' and h.purpose <> 'Retesting for ART initiation'
                            )
                            SELECT * FROM ranked_messages WHERE rnt = 1
                        """
                        ).df()
        
        df_hts_from_art = duckdb.query(
                        """
                            select date_of_hiv_test as 'Event_date',
                            person_id, 
                            date_of_hiv_test as 'Date of HIV Test',
                            'POSITIVE' as HIV_Result 
                            from art
                            where person_id not in (select person_id from df_hts)
                        """
                        ).df()
        
        df_hts_from_art["Date of HIV Test"] = pd.to_datetime(df_hts_from_art["Date of HIV Test"],errors='coerce').dt.date
        df_hts["Date of HIV Test"] = pd.to_datetime(df_hts["Date of HIV Test"],errors='coerce').dt.date
        df_hts_from_art["Event_date"] = pd.to_datetime(df_hts_from_art["Event_date"],errors='coerce').dt.date
        df_hts["Event_date"] = pd.to_datetime(df_hts["Event_date"],errors='coerce').dt.date
        df_positives = pd.merge(df_hts,df_hts_from_art,how = "outer")
        
        if df_positives.empty:
                print("    >>> There are no positive cases in the database")
                continue
        
        # ---------------a1. new HIV positives
        df_hts_investigations =  duckdb.query(
                        """
                            WITH ranked_messages AS(
                            select  h.laboratory_investigation_id, h.result,h.time , p.person_investigation_id, p.person_id,
                            ROW_NUMBER() OVER (PARTITION BY p.person_investigation_id ORDER BY time ASC) as rtt
                            from person_investigation p
                            left join laboratory_investigation_test h
                            on p.person_investigation_id = h.laboratory_investigation_id
                            )
                            select * from ranked_messages 
                            where person_investigation_id in 
                            ( select person_investigation_id from df_hts)
                        """
                        ).df()
            
        df_hts_investigations['rtt'] = df_hts_investigations['rtt'].map({1:"HIVtestOneResult", 2: "HIVtestTwoResult",3:"HIVtestThreeResult"})
        df_hts_investigations = df_hts_investigations.pivot(index='person_id', columns=['rtt'], values='result')
        df_hts_investigations = df_hts_investigations.reset_index()

        # ---------------a2. new HIV positives
        df_sexual_history = duckdb.query(
                        """
                            select sexual_history_id,person_id,sexually_active,date from sexual_history
                        """
                        ).df()
        replace_dict = {1:"YES",0:"NO"}
        sexual_history.replace({"sexually_active": replace_dict},inplace=True)
        df_sexual_history= sexual_history[['sexual_history_id', 'person_id',  'sexually_active','date']]


        # ---------------a3. new HIV positives
        df_sexual_history_question = duckdb.query(
                        """
                            select sexual_history_id,question,response_type from sexual_history_question 
                        """
                        ).df()
        df_sexual_history_question.drop_duplicates(subset=["sexual_history_id","question"],inplace = True)
        df_sexual_history_question = df_sexual_history_question.pivot(index='sexual_history_id', columns='question')
        df_sexual_history_question.columns = df_sexual_history_question.columns.droplevel()
        df_sexual_history_question.reset_index(inplace=True)
        df_sexual_history_question = pd.merge(df_sexual_history_question, df_sexual_history , how="left", on=["sexual_history_id"])
        df_sexual_history_question.drop(['sexual_history_id'], axis=1,inplace=True)
        df_sexual_history_question.rename(columns = {'Exchanged sex for  money/material goods':'Exchanged sex for moneymaterial goods',
                                                        'Victim/ Suspected victim of sexual abuse':'Victim Suspected victim of sexual abuse',
                                                        'Unprotected sex without a condom':'Unprotected sex without a condom',
                                                        'sexually_active': 'Sexually Active' }, inplace = True)

        column_order = {'Event_date': 1, 
                            'person_id':2,'Been incarcerated into jail':3,
                    'Exchanged sex for moneymaterial goods':4,
                    'Had Anal Sex':5,'Had sex with male':6,'Had sex with female':7,'Had sex with a sex worker':8,
                    'Victim Suspected victim of sexual abuse':9,'Tattooed with unsterilized instruments':10,
                    'Received medical injections':11, 'Unprotected sex without a condom':12, 'Injected recreational drugs':13,
                    'History of an STI':14, 'Had Oral Sex':15,  'Received blood transfusions':16, 'Sexually Active':17
                            }
        df_sexual_history_question = df_sexual_history_question[[col for col in sorted(df_sexual_history_question.columns, key=lambda x: column_order.get(x, float('inf')))]]
        df_sexual_history_question.drop_duplicates(subset=['person_id'], keep='last',inplace=True)
        
        df_hts_positives_merge = pd.merge(df_positives,df_hts_investigations, on =["person_id"], how ="left")
        df_hts_positives = pd.merge(df_hts_positives_merge,df_sexual_history_question, on =["person_id"] , how = "left")
        df_hts_positives.insert(0,"Facility Id",facility_id)
        df_hts_positives.rename(columns = {"HIV_Result":"Final HIV Result" ,
                                        "reason_for_not_initiating_art":"Reason for not initiating on ART"  },inplace = True)
        
        df_hts_positives.replace(replace_dict, inplace=True)
        df_hts_positives["Date of HIV Test"] = pd.to_datetime(df_hts_positives["Date of HIV Test"],errors='coerce').dt.date
        df_hts_positives = df_hts_positives.sort_values(by=['Final HIV Result'])
        df_hts_positives.drop_duplicates(subset=['Event_date', 'HTS Number','person_id','Date of HIV Test'], keep='last',inplace=True)
        global_df_hts_positives = extraction.concat_dataframes([global_df_hts_positives,df_hts_positives])


        #---------------b. demographics
        df_demographics_extract = duckdb.query(
                        """
                            select 
                            person_id,firstname,lastname as Lastname, birthdate, sex, 
                            self_identified_gender as 'Self Identified Gender',
                            marital_id as 'Marital Id', marital as 'Marital Status',education_id as 'Education Id',
                            education as 'Education',occupation_id as 'Occupation Id' ,occupation as 'Occupation',
                            religion_id as 'Religion id', religion as 'Religion',nationality_id as 'Nationality Id',
                            nationality as Nationality, street as 'Street',city as City, town_id as 'Residential Town Id', 
                            town as 'Residential Town'
                            from demographics where person_id in (select person_id from df_hts_positives)
                        """
            ).df()
        
        # df_demographics_extract = df_demographics_extract.sort_values(by=['birthdate'])
        # df_demographics_extract.drop_duplicates(subset=['person_id'], keep='last',inplace = True)
        df_patient_client_profile = duckdb.query(
                        """
                            select 
                            person_id, client_profile as 'Client Profile'
                            from patient_client_profile
                            where person_id in (select person_id from df_hts_positives)
                        """
                ).df()
        df_patient_client_profile.drop_duplicates(subset=['person_id'], keep='last',inplace = True)
        df_demographics = pd.merge(df_demographics_extract,df_patient_client_profile,on =["person_id"],how="left")
        df_demographics.insert(0,"Facility Id",facility_id)
        global_df_demos = extraction.concat_dataframes([global_df_demos,df_demographics])


        #---------------c. Extract date of all known hiv negative test from hts
        df_hts_negative = duckdb.query(
                        """
                            select 
                            'hts' as 'Source of Last Negative',
                            pt.time as Event_date,
                            p.person_id,
                            pt.time as 'Last HIV negative date'
                            FROM hts h  
                            left join patient pt
                            on  h.patient_id = pt.patient_id
                            left join person_investigation p
                            on h.laboratory_investigation_id = p.person_investigation_id
                            where  p.test = 'HIV' and p.result in ('Negative' ,'NEGATIVE' ,'NEG','Neg')
                           
                        """
                        ).df()
        
        df_hts_screening_negative = duckdb.query(
                        """
                            select 
                            'HTS Screening' as 'Source of Last Negative',
                            h.date_last_tested as Event_date,
                            p.person_id ,
                            h.date_last_tested as 'Last HIV negative date'
                            from hts_screening h
                            left join patient p
                            on h.patient_id = p.patient_id
                            where result in ('Negative' ,'NEGATIVE' ,'NEG','Neg')
                        """
                        ).df()
        
        df_investigation_negative = duckdb.query(
                        """
                            select 
                            'Investigation' as 'Source of Last Negative',
                            date as Event_date,
                            person_id,
                            date as 'Last HIV negative date'
                            FROM person_investigation 
                            where  test = 'HIV' and result in ('Negative' ,'NEGATIVE' ,'NEG','Neg')  
                            and (person_id not in (select person_id from df_hts_negative)
                            or
                            person_id not in (select person_id from df_hts_screening_negative))
                           
                        """
                        ).df()
        
        

        df_negatives = pd.concat([df_hts_negative,df_investigation_negative,df_hts_screening_negative])
        
        df_negatives["Event_date"] = pd.to_datetime(df_negatives["Event_date"],errors='coerce').dt.date
        df_negatives["Last HIV negative date"] = pd.to_datetime(df_negatives["Last HIV negative date"],errors='coerce').dt.date
        df_negatives.insert(0,"Facility Id",facility_id)
        global_df_hts_negative = extraction.concat_dataframes([global_df_hts_negative,df_negatives])

        #---------------d. cbs
        df_cbs = duckdb.query(
        """
                select date as 'Event_date' ,
                person_id,'Yes' as 'Notified', date as 'Date of Notification' , been_on_prep as 'Been On Prep' 
                from cbs 
        
        """
        ).df()
        df_cbs.insert(0,"Facility Id",facility_id)
        df_cbs = df_cbs[[
            'Event_date', "Facility Id",'Been On Prep',
            "person_id", 'Notified',
            'Date of Notification'
        ]]
        df_cbs["Date of Notification"] = pd.to_datetime(df_cbs["Date of Notification"],errors='coerce').dt.date
        df_cbs["Event_date"] = pd.to_datetime(df_cbs["Event_date"],errors='coerce').dt.date
        df_cbs.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_cbs = extraction.concat_dataframes([global_df_cbs,df_cbs])

        #---------------e. recency
        df_recency = duckdb.query(
                        """
                            select date as 'Event_date',
                            person_id, date as 'Date Recency Done',
                            result as 'Recency Result'
                            from person_investigation 
                            where test = 'HIV-1 Rapid Recency' 
                        """
                        ).df()
        df_recency["Date Recency Done"] = pd.to_datetime(df_recency["Date Recency Done"],errors='coerce').dt.date
        df_recency["Event_date"] = pd.to_datetime(df_recency["Event_date"],errors='coerce').dt.date
        df_recency.insert(0,"Facility Id",facility_id)
        df_recency = df_recency [[
            'Event_date', "Facility Id",
            "person_id",
            'Date Recency Done' ,'Recency Result'
        ]]
        df_recency = df_recency.sort_values(by=['Recency Result'])
        df_recency.drop_duplicates(subset=['Event_date','person_id'], keep='first',inplace = True)
        global_df_recency = extraction.concat_dataframes([global_df_recency,df_recency])

        #---------------f. art
        df_art = duckdb.query(
                    """
                        select  date as 'Event_date',
                        art_id,person_id,art_number as "Art Number",art_cohort_number as "Art Cohort Number",
                        date as "Art_Initiation_Date",date_enrolled as "Date Enrolled into ART"
                        from art 
                    """
                    ).df()
        df_art.insert(0,"Facility Id",facility_id)
        df_art["Art_Initiation_Date"] = pd.to_datetime(df_art["Art_Initiation_Date"],errors='coerce').dt.date
        df_art["Event_date"] = pd.to_datetime(df_art["Event_date"],errors='coerce').dt.date
        df_art = df_art [[
            'Event_date', "Facility Id",
            "person_id",
            "Art Number", "Art Cohort Number", "Art_Initiation_Date","Date Enrolled into ART"
        ]]
        df_art = df_art.sort_values(by=['Art Number'])
        df_art.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_art = extraction.concat_dataframes([global_df_art,df_art])

        #---------------g. art_visit
        df_art_visit = duckdb.query(
                """
                    select p.time as 'Event_date', 
                    p.time as "Art visit date", p.person_id, a.visit_type as 'Art visit type',
                    a.tb_status as 'TB Status', a.family_planning_status, a.lactating_status
                    from art_visit a left join patient p
                    on a.patient_id = p.patient_id
                    
                """
                ).df() 
        df_art_visit.insert(0,"Facility Id",facility_id)
        df_art_visit["Art visit date"] = pd.to_datetime(df_art_visit["Art visit date"],errors='coerce').dt.date
        df_art_visit["Event_date"] = pd.to_datetime(df_art_visit["Event_date"],errors='coerce').dt.date
        df_art_visit = df_art_visit[[
                'Event_date', "Facility Id",
                'person_id',
                "Art visit date",'Art visit type',
                'TB Status'
            ]]
        df_art_visit = df_art_visit.sort_values(by=['Art visit type'])
        df_art_visit.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        df_art_visit = df_art_visit.replace('Presumptive-if there are signs', 'Suspect or Presumptive')
        df_art_visit = df_art_visit.replace('Screened and has no signs', 'Screened with no signs')
        df_art_visit = df_art_visit.replace('Tb status not assesssed', '')
        global_df_art_visit = extraction.concat_dataframes([global_df_art_visit,df_art_visit])

        #---------------h. art current status
        df_art_current_status = duckdb.query(
                """
                    select a.date as 'Event_date', 
                    a.date as 'Art Current Status Date',a.art_id ,a.regimen as regimen , a.state as 'ARV Status', a.art_initiation_category as 'Art Initiation Category',
                    art.person_id,a.regimen_id as 'Regimen Id'
                    from art_current_status a 
                    left join art 
                    on a.art_id = art.art_id
                    
                """
                ).df()
        df_art_current_status.insert(0,"Facility Id",facility_id)
        df_art_current_status["Art Current Status Date"] = pd.to_datetime(df_art_current_status["Art Current Status Date"],errors='coerce').dt.date
        df_art_current_status["Event_date"] = pd.to_datetime(df_art_current_status["Event_date"],errors='coerce').dt.date
        df_art_current_status = df_art_current_status[[
        'Event_date',"Facility Id",'person_id', 'ARV Status','Regimen Id','regimen','Art Initiation Category'
        ]]
        df_art_current_status = df_art_current_status.sort_values(by=['regimen'])
        df_art_current_status.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_art_current_status = extraction.concat_dataframes([global_df_art_current_status,df_art_current_status])

        #---------------i. art who stage
        df_art_who_stage = duckdb.query(
                """
                    select  a.date as 'Event_date',
                    p.person_id , a.art_id, a.date as 'Who Stage Date',
                    a.date as 'Outcome Date',
                    a.stage as 'Who Stage', a.follow_up_status as 'Art Outcome'
                    from art_who_stage a left join art p
                    on a.art_id  = p.art_id
                    
                """
            ).df()
        df_art_who_stage.insert(0,"Facility Id",facility_id)
        df_art_who_stage["Who Stage Date"] = pd.to_datetime(df_art_who_stage["Who Stage Date"],errors='coerce').dt.date
        df_art_who_stage["Event_date"] = pd.to_datetime(df_art_who_stage["Event_date"],errors='coerce').dt.date
        df_art_who_stage = df_art_who_stage [[
            'Event_date', "Facility Id",
            'person_id','Who Stage Date','Who Stage',
            'Art Outcome'
        ]]
        df_art_who_stage = df_art_who_stage.sort_values(by=['Who Stage'])
        df_art_who_stage.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_art_who_stage = extraction.concat_dataframes([global_df_art_who_stage,df_art_who_stage])

        #---------------j. viral load
        df_viral_load = duckdb.query(
                """
                    select date as "Event_date",
                    person_id,date as "Date at which Viral Load result was issued",
                    date as "Date for which Viral Load was taken",
                    'TRUE' as "Viral Load Sample submitted to lab",
                    'TRUE' as "Was Viral Load result issued",
                    result as "Viral Load result",
                    from person_investigation where test = 'Viral Load'
                    
                """
            ).df()
        df_viral_load.insert(0,"Facility Id",facility_id)
        df_viral_load["Date at which Viral Load result was issued"] = pd.to_datetime(df_viral_load["Date at which Viral Load result was issued"],errors='coerce').dt.date
        df_viral_load["Event_date"] = pd.to_datetime(df_viral_load["Event_date"],errors='coerce').dt.date
        df_viral_load = df_viral_load[[
                    "Event_date","Facility Id",
                    'person_id',
                    "Date at which Viral Load result was issued",
                    "Date for which Viral Load was taken",
                    "Viral Load Sample submitted to lab",
                    "Was Viral Load result issued",
                    "Viral Load result"
        ]]
        df_viral_load = df_viral_load.sort_values(by=['Viral Load result'])
        df_viral_load.drop_duplicates(subset=['Event_date','person_id'], keep='last', inplace = True)
        global_df_viral_load = extraction.concat_dataframes([global_df_viral_load,df_viral_load])

        #---------------k. cd4
        df_cd4 = duckdb.query(
                """
                    select date as "Event_date",
                    person_id, date as 'Date at which cd4 sample was taken',
                    date as 'Date at which cd4 result was issued',
                    'TRUE' as "CD4 Sample submitted to lab",
                    'TRUE' as "Was cd4 result issued",
                    result as 'CD4 Count'
                    from person_investigation where test = 'CD4 Count'
                  
                """
                ).df()
        df_cd4.insert(0,"Facility Id",facility_id)
        df_cd4["Date at which cd4 result was issued"] = pd.to_datetime(df_cd4["Date at which cd4 result was issued"],errors='coerce').dt.date
        df_cd4["Event_date"] = pd.to_datetime(df_cd4["Event_date"],errors='coerce').dt.date
        df_cd4 = df_cd4 [[
            'Event_date',"Facility Id",
            'person_id',
            'Date at which cd4 sample was taken',
            'Date at which cd4 result was issued',
            "CD4 Sample submitted to lab",
            "Was cd4 result issued",
            'CD4 Count',
        ]]
        df_cd4 = df_cd4.sort_values(by=['CD4 Count'])
        df_cd4.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_cd4 = extraction.concat_dataframes([global_df_cd4,df_cd4])

        #---------------l. TB
        df_TB = duckdb.query(
                """
                    select date as 'Event_date',
                    person_id,date as 'TB Treatment Start Date',type_of_tb as 'Type Of TB',
                    tb_disease_site as 'TB Disease Site',
                    tb_disease_type as 'TB Disease Type',outcome as 'TB Outcome'
                    from tb
                    
                """
            ).df()
        df_TB.insert(0,"Facility Id",facility_id)
        df_TB["TB Treatment Start Date"] = pd.to_datetime(df_TB["TB Treatment Start Date"],errors='coerce').dt.date
        df_TB["Event_date"] = pd.to_datetime(df_TB["Event_date"],errors='coerce').dt.date
        df_TB = df_TB[[
                'Event_date',"Facility Id",
                'person_id','TB Treatment Start Date',
                'TB Outcome','Type Of TB','TB Disease Site','TB Disease Type'
            ]]
        df_TB = df_TB.sort_values(by=['TB Outcome'])
        df_TB.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
        global_df_tb = extraction.concat_dataframes([global_df_tb,df_TB])

        #---------------m. TB screening
        df_TB_Screening = duckdb.query(
                """
                    select t.time as 'Event_date',
                    t.person_id , p.presumptive as 'TB Screened', 
                    from tb_screening p
                    left join patient t 
                    on p.patient_id = t.patient_id
                    
                """
                ).df()
        df_TB_Screening.insert(0,"Facility Id",facility_id)
        df_TB_Screening = df_TB_Screening.replace('\x01', 'Suspect or Presumptive')
        df_TB_Screening = df_TB_Screening.replace('\\0', 'Screened with no signs')
        df_TB_Screening = df_TB_Screening[[
            'Event_date', "Facility Id",
            'person_id',
            'TB Screened'
        ]]
        df_TB_Screening["Event_date"] = pd.to_datetime(df_TB_Screening["Event_date"],errors='coerce').dt.date
        df_TB_Screening.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
        global_df_tb_screening = extraction.concat_dataframes([global_df_tb_screening,df_TB_Screening])

        #---------------n. transfer out
        df_transfer_out = duckdb.query(
                    """
                    select 
                    a.person_id , t.art_id , t.transfer_out_date as 'Event_date',
                    t.transfer_out_date as 'Transfer Out date',
                    t.transfer_reason as 'Transfer Reason', t.transfer_facility_id
                    from art_transfer_out t left join art a
                    on t.art_id = a.art_id
                """
                ).df()
        df_transfer_out["Event_date"] = pd.to_datetime(df_transfer_out["Event_date"],errors='coerce').dt.date
        df_transfer_out = df_transfer_out.sort_values(by=['transfer_facility_id'])
        df_transfer_out.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        df_transfer_out.insert(0,"Facility Id",facility_id)
        global_df_transfer_out = extraction.concat_dataframes([global_df_transfer_out,df_transfer_out])

    # -------------Section2 -------------------------------------- -------------------------------------
    #---------------a. deduplication
    df_demographics_for_dedup = global_df_demos[["Facility Id",
            "person_id", "firstname", "Lastname", "birthdate", "sex",
            'Self Identified Gender', 'Street',
            "City", 'Education Id', 'Occupation Id','Religion id'
            ]]
    
    global_df_demos.to_csv("Demographics original.csv")
    
    df_demographics_for_dedup["birthdate"] = pd.to_datetime(df_demographics_for_dedup["birthdate"],errors='coerce').dt.date
    df_demographics_for_dedup = df_demographics_for_dedup.sort_values(by=['birthdate'])
    df_demographics_for_dedup.drop_duplicates(subset=['person_id'], keep='last',inplace=  True)
    df_demographics_for_dedup.fillna("NULL",inplace=True)
    clusters = extraction.deduplication(df_demographics_for_dedup)

    print("    >>> Generating CBS ID")
    #---------------b. Generate CBS ID 
    cluster_id_dictionary = {}
    seq_number = 0
    
    
    cbs_ids =[]
    for i,row in clusters.iterrows():
        if row["cluster id"] in cluster_id_dictionary.keys() or row["firstname"] == "No Name":
            seq_number = seq_number
        else:
            seq_number = seq_number + 1
            extraction.cbs_unique_id(row,seq_number)
        cbs_ids.append(extraction.cbs_unique_id(row,seq_number))
    clusters["CBS ID"] = cbs_ids
    clusters.sort_values(by = ["cluster id"],inplace = True)

    person_id_cbs_id = clusters [[
        "person_id","CBS ID"
    ]]
    dictionary_mapping = person_id_cbs_id.set_index('person_id')['CBS ID'].to_dict()
  
    print("    >>> Add CBS ID to all dataframes")
    #-------------- c. Add CBS ID to every global dataframe
    list_of_globals = [global_df_demos, global_df_hts_positives,global_df_hts_negative,
              global_df_cbs, global_df_recency, global_df_art,
              global_df_art_visit, global_df_art_current_status, global_df_viral_load,
              global_df_cd4,global_df_art_who_stage,global_df_tb,global_df_tb_screening,
              global_df_transfer_out
              ]
    for df in list_of_globals:
        df['CBS ID'] = df['person_id'].map(dictionary_mapping)
    
    #----------------c2 Add demographics to dataset with positives
    global_df_demos.drop(['Facility Id'], axis=1, inplace=True)
    merge_demos = pd.merge(global_df_hts_positives,global_df_demos, on =["person_id","CBS ID"] , how = "left")
    merge_demos.sort_values(["CBS ID"],inplace=True)

    #---------------d. Merging files on CBS ID
    # --------------i. Merge with negatives....................
    # Remove people who do not have positive test or art initiation
    print("    >>> Merging dataframes")
    global_negatives = extraction.remove_people_without_positive_result(global_df_hts_negative,global_df_hts_positives)
    global_negatives_sorted = global_negatives.sort_values(by=['Source of Last Negative']).drop_duplicates(['person_id','Last HIV negative date'], keep='first')

    merge_demos["Event_date"] = pd.to_datetime(merge_demos["Event_date"],errors='coerce').dt.date

    merge_negatives1  = extraction.get_last_negative_date(global_negatives_sorted, merge_demos)
    merge_negatives1.drop(['person_id'], axis=1, inplace=True)
    merge_negatives2 = pd.merge(merge_demos, merge_negatives1, on =["Event_date","CBS ID","Facility Id"] , how = "outer")

    merge_negatives2.sort_values(["CBS ID","Event_date"],inplace=True)
    
    #---------------ii. Merge with recency....................
    global_recency = extraction.remove_people_without_positive_result(global_df_recency,global_df_hts_positives)
    global_recency = global_recency.sort_values(by=['Event_date']).drop_duplicates(['person_id'], keep='first')
    global_recency["Event_date"] = pd.to_datetime(global_recency["Event_date"],errors='coerce').dt.date
    global_recency.drop(['person_id'], axis=1, inplace=True)
    merge_recency = pd.merge(merge_negatives2, global_recency, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_recency.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------iii Merge with art data..................
    global_art = extraction.remove_people_without_positive_result(global_df_art,global_df_hts_positives)
    global_art = global_art.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_art["Event_date"] = pd.to_datetime(global_art["Event_date"],errors='coerce').dt.date
    global_art.drop(['person_id'], axis=1, inplace=True)
    merge_art = pd.merge(merge_recency,global_art, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_art.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------iv Merge with art visit..................
    global_art_visit = extraction.remove_people_without_positive_result(global_df_art_visit,global_df_hts_positives)
    global_art_visit = global_art_visit.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_art_visit["Event_date"] = pd.to_datetime(global_art_visit["Event_date"],errors='coerce').dt.date
    global_art_visit.drop(['person_id'], axis=1, inplace=True)
    merge_art_visit = pd.merge(merge_art,global_art_visit, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_art_visit.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------v Merge with art current..................
    global_art_current_status = extraction.remove_people_without_positive_result(global_df_art_current_status,global_df_hts_positives)
    global_art_current_status = global_art_current_status.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_art_current_status["Event_date"] = pd.to_datetime(global_art_current_status["Event_date"],errors='coerce').dt.date
    global_art_current_status.drop(['person_id'], axis=1, inplace=True)
    merge_art_current = pd.merge(merge_art_visit,global_art_current_status, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_art_current.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------vi Merge with who stage..................
    global_art_who_stage = extraction.remove_people_without_positive_result(global_df_art_who_stage,global_df_hts_positives)
    global_art_who_stage = global_art_who_stage.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_art_who_stage["Event_date"] = pd.to_datetime(global_art_who_stage["Event_date"],errors='coerce').dt.date
    global_art_who_stage.drop(['person_id'], axis=1, inplace=True)
    merge_who = pd.merge(merge_art_current,global_art_who_stage, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_who.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------vii Merge with viral load..................
    global_viral_load = extraction.remove_people_without_positive_result(global_df_viral_load,global_df_hts_positives)
    global_viral_load = global_viral_load.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_viral_load["Event_date"] = pd.to_datetime(global_viral_load["Event_date"],errors='coerce').dt.date
    global_viral_load.drop(['person_id'], axis=1, inplace=True)
    merge_vl = pd.merge(merge_who,global_viral_load, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_vl.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------viii Merge with cd4..................
    global_cd4 = extraction.remove_people_without_positive_result(global_df_cd4,global_df_hts_positives)
    global_cd4 = global_cd4.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_cd4["Event_date"] = pd.to_datetime(global_cd4["Event_date"],errors='coerce').dt.date
    global_cd4.drop(['person_id'], axis=1, inplace=True)
    merge_cd4 = pd.merge(merge_vl,global_cd4, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_cd4.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------ix Merge with tb..................
    global_tb = extraction.remove_people_without_positive_result(global_df_tb,global_df_hts_positives)
    global_tb = global_tb.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_tb["Event_date"] = pd.to_datetime(global_tb["Event_date"],errors='coerce').dt.date
    global_tb.drop(['person_id'], axis=1, inplace=True)
    merge_tb = pd.merge(merge_cd4,global_tb, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_tb.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------x Merge with tb screening..................
    global_tb_screening = extraction.remove_people_without_positive_result(global_df_tb_screening,global_df_hts_positives)
    global_tb_screening = global_tb_screening.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_tb_screening["Event_date"] = pd.to_datetime(global_tb_screening["Event_date"],errors='coerce').dt.date
    global_tb_screening.drop(['person_id'], axis=1, inplace=True)
    merge_tb_screening = pd.merge(merge_tb,global_tb_screening, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_tb_screening.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------xi Merge with tb screening..................
    global_transfer_out = extraction.remove_people_without_positive_result(global_df_transfer_out,global_df_hts_positives)
    global_transfer_out = global_transfer_out.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_transfer_out["Event_date"] = pd.to_datetime(global_transfer_out["Event_date"],errors='coerce').dt.date
    global_transfer_out.drop(['person_id'], axis=1, inplace=True)
    merge_transfer_out = pd.merge(merge_tb_screening,global_transfer_out, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_transfer_out.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------xii Merge with cbs..................
    global_cbs = extraction.remove_people_without_positive_result(global_df_cbs,global_df_hts_positives)
    global_cbs = global_df_cbs.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_cbs["Event_date"] = pd.to_datetime(global_cbs["Event_date"],errors='coerce').dt.date
    global_cbs.drop(['person_id'], axis=1, inplace=True)
    merge_cbs = pd.merge(merge_transfer_out,global_cbs, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_cbs.sort_values(["CBS ID","Event_date"],inplace=True)
    merge_cbs.rename(columns = {'Event_date':'Event date'}, inplace = True)

    merge_cbs = merge_cbs [['person_id',
            'CBS ID','Event date', 'Facility Id',
            'Client Profile', 'birthdate', 'sex', 'Self Identified Gender', 
            'Marital Status',  'Education','Occupation', 'Religion', 
            'Nationality', 'Residential Town', 'Date of HIV Test', 
            'Reason for HIV Test',
            'Entry Point', 'Pregnant during HIV Test',
            'Already positive', 'Already on art', 'Test Method', 
            'Source of Last Negative','Last HIV negative date',
            'HIVtestOneResult','HIVtestTwoResult', 'HIVtestThreeResult','Final HIV Result',
            'Been incarcerated into jail',
            'Exchanged sex for moneymaterial goods', 'Had Anal Sex',
            'Had sex with male', 'Had sex with female', 'Had sex with a sex worker',
            'Victim Suspected victim of sexual abuse',
            'Tattooed with unsterilized instruments', 'Received medical injections',
             'Injected recreational drugs',
            'History of an STI', 'Had Oral Sex', 'Received blood transfusions',
            'Sexually Active', 'Notified','Date of Notification', 'Been On Prep',
            'Date Recency Done', 'Recency Result','Reason for not doing recency test',
            'Art Number',
            'Art Cohort Number', 'Art_Initiation_Date', 'Date Enrolled into ART','Reason for not initiating on ART',
            'Art visit date', 'Art visit type',
            'ARV Status', 'regimen',
            'Art Initiation Category','Who Stage Date', 'Who Stage', 
            'Date for which Viral Load was taken',
            'Viral Load Sample submitted to lab',
            'Was Viral Load result issued',
            'Date at which Viral Load result was issued',
            'Viral Load result',
            'Date at which cd4 sample was taken',
            'Date at which cd4 result was issued', 'CD4 Sample submitted to lab',
            'Was cd4 result issued',
            'CD4 Count','TB Treatment Start Date',
            'TB Outcome', 'Type Of TB', 'TB Disease Site', 'TB Disease Type',
            'Art Outcome',
            'Transfer Out date',
            'Transfer Reason',
            'transfer_facility_id'
    ]]
    
    filename = datetime.today().strftime('%d%m%Y')
    merge_cbs.to_csv("CBS_" + filename + ".csv",index=False)

In [5]:
import json
import os
import re
from pathlib import Path
import pandas as pd
import pandas_dedupe
import duckdb
import mysql.connector
import subprocess
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

class CBSExtraction:

    def __init__(self, path_to_databases,path_to_json):
        self.folder_path = path_to_databases
        self.json_path = path_to_json
    
    def get_connection(self):
        connection = mysql.connector.connect(user="root", password="root", host="localhost", port="3306")
        return connection


    def extracting_data(self,sql,connection):
        connection = self.get_connection()
        df = pd.read_sql(sql,connection)
        connection.close()
        return df

    def get_facility_databases(self,folder_path): 
        facility_databases = []
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.endswith(".sql"):
                    file_path = os.path.join(root, file)
                    facility_databases.append(file_path)
        return facility_databases

    def restore_database(self,backup_file,time_):
        backup_file = backup_file.replace("\\", "/")
        # FIRST CHECK FOR EXISTING DATABASES AND DROP THEM
        connection = self.get_connection()

        results = pd.read_sql('SELECT schema_name FROM information_schema.schemata;', con=connection)
        schemas = list(results["schema_name"])
        # Create a cursor object
        cursor = connection.cursor()
        # Execute a query
        cursor.execute(f'SET foreign_key_checks = 0')
        for schema in schemas:
            if schema in ['client', 'consultation', 'deduplication', 'facility', 'mrs', 'provider', 'report', 'terminology', 'zimepms']:
                results = cursor.execute(f'DROP DATABASE {schema}')
                print("DROPPED [{0}]".format(schema))
        cursor.close()
        restore_command = f'mysql -u root -proot -f < {backup_file}'
        try:
            print(">>> Restoring database")
            subprocess.run(restore_command, shell=True, check=True)
            print(f'DATABASE RESTORE [SUCCESSFUL>>>]')
        except Exception as e:
            with open('./logs.txt', 'a') as f:
                f.write(time_ + f'{backup_file}\n')

    @staticmethod
    def get_facility_identifiers(metadata):
        """Returns the siteId from the metadata string."""
        siteId = re.search(r"<string>siteId</string><string>(.*?)</string>", metadata)
        version = re.search(r"<string>version</string><string>(.*?)</string>", metadata)
        last_time_stamp = re.search(r"<string>time</string><string>(.*?)</string>", metadata)
        if siteId:
            return siteId.group(1) , version.group(1) , last_time_stamp.group(1)
        else:
            return "","",""
    
    @staticmethod
    def get_first_time_stamp(metadata):
        """Returns the siteId from the metadata string."""
        first_time_stamp = re.search(r"<string>time</string><string>(.*?)</string>", metadata)
        if first_time_stamp:
            return first_time_stamp.group(1) 
        else:
            return ""
        
    @staticmethod
    def get_size_of_folder(filename):
        folder_size = round(os.path.getsize(filename)/(1000*1024),2)
        return folder_size
    

    def split_string(self,string):
        # Match all commas outside of quotes
        pattern = r",(?=(?:[^']*'[^']*')*[^']*$)"
        return re.split(pattern, string)
    
    def save_json(self,dictionary,filename):
        with open(self.json_path + filename + ".json", "w") as fp:
            json.dump(dictionary,fp)

    def read_json(self,facility_id):
        with open(self.json_path + facility_id + ".json", "r") as fp:
            data = json.load(fp)
        facility_id = list(data.keys())[0]
        return data , facility_id
    
    
    def construct_table_data(self,filename):
        with open(os.path.join(self.folder_path, filename), "r", encoding="ISO-8859-1") as f:
            content = f.read()
        # content = content.replace("Unprotected sex (sex without a condom)", "Unprotected sex without a condom"
        content = re.sub(r"\)'", "", content)
        subText = "',',"
        content = content.replace(subText,"','")
        latest_site_id = []
        latest_timestamp = []
        first_timestamp = []
        version = []
        
        domain_event = ""
        if "/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;" in content:
            domain_event = content.split("/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;")
        else:
            latest_site_id = "corrupt"
            fact_tables = {}
            row = ""
            first_timestamp =  ""
            version = ""
            return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables

        if "/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */" in domain_event[1]:
            domain_event = domain_event[1].split("/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */")
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[0])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[0])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[0])
        else:
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[1])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[1])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[1])
        if facility_search:
            latest_site_id = facility_search[-1]
            latest_timestamp = time_serach[-1]
            first_timestamp = time_serach[0]
            if len(version)==0:
                version = ""
            else:
                version = version[-1]
        db_tables = {}
        fact_tables = {}
        database_statements = re.split(r'(?<=\n)CREATE DATABASE', content)
        for statement in database_statements[1:]:
            table_names = []
            database_name = re.findall(r'`(\w+)`', statement)[0]

            if database_name not in ("client","consultation","report"):
                 continue
            # Use regular expressions to extract the table names
            pattern = re.compile(r"CREATE TABLE `(.+?)`", re.DOTALL)
            matches = pattern.findall(statement)
            
            # Loop through the matches and append the table names to the list
            for match in matches:
                table_names.append(match)

            
            # Create a dictionary of dataframes for the tables
            tables_dict = {}
            for table_name in table_names:

                # if table_name not in table_list:
                #     continue
                table_str = re.search(rf'CREATE TABLE `{table_name}`.+?ENGINE=InnoDB DEFAULT CHARSET', statement, re.DOTALL)
                if table_str is None:
                    continue
                table_str  = table_str.group(0).strip()
                columns = re.findall(r'`(\w+)` (\w+)', table_str)
                column_names = [column[0] for column in columns if not column[0].startswith("fk")]
                data = re.findall(rf'INSERT INTO `{table_name}` VALUES \(.+?\);', statement, re.DOTALL)
                rows = {}
                if data:
                    values_list = []
                    for row in data: 

                        row = re.findall(r'\((.*?)\)', row)
                        # row = row.split('(',1)[1]
                        
                        values2 = [ re.split(r',(?=")', s) for s in row]
            
                        for item in values2:
                            item = item[0].replace('NULL', "'NULL'")
                            rec = [i.replace("'","").lstrip() for i in item.split("',")]
                            values_list.append(rec)
                     
                    for column in column_names:
                        coulum_data = []
                        column_index = column_names.index(column)
                        for record in values_list:
                            if column_index < len(record):
                                coulum_data.append(record[column_index])
                            else:
                                coulum_data.append("")  
                        rows[column] = coulum_data
                    table_df = pd.DataFrame(rows)
                    table_df = table_df.to_dict(orient="records")
                    tables_dict[table_name] = table_df
                # else:
                #     table_df = pd.DataFrame(columns=column_names)
                #     tables_dict[table_name] = table_df
                # Add the database and its tables to the dictionary
            db_tables[database_name] = tables_dict

            if len(latest_site_id)==0:
                latest_site_id = ""
                fact_tables = {}
                row = ""
                first_timestamp =  ""
                version = ""
                return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables
            
        fact_tables[latest_site_id] = db_tables

        return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables
    
    def get_database_size(self,filename):
        size = round(Path(self.folder_path + filename).stat().st_size /(1000*1024),2)
        return size
    
    @staticmethod
    def log(processing_time,province="",district="",facility="",site_code="",filename="",first_date="",last_date="",
            db_size="",version="",comment="",ehr_type = ""): 
         df = pd.DataFrame({"Time File Processed":[processing_time],
                            "province": [province],
                            "district": [district],
                            "facility": [facility],
                            "Site Code":[site_code],
                            "File Name":[filename],
                            "Date site first used EHR": [first_date],
                            "Date site last used EHR": [last_date],
                            "Database Size": [db_size],
                            "Version":[version],
                            "Comments":[comment],
                            "EHR Type":[ehr_type]
                            })
         df.to_csv('log.csv', mode='a', index = False, header=None)
    
    @staticmethod
    def dbTableList():
        # List of tables to use for the program / this reduce time for table reconstruction by focusing only on tables to be used
        table_list = ["hts","hts_screening","person_investigation","patient","cbs",
                      "sexual_history","sexual_history_question","art","art_visit",
                      "art_current_status","tb","art_who_stage","person",
                      "art_appointment","laboratory_investigation_test","laboratory_request_order",
                      "art_transfer_out","patient_tb_screening","patient_client_profile"]
        return table_list
    
    @staticmethod
    def get_mapping_file():
        mapping_file = pd.read_csv("mapping_file.csv")
        mapping_file['Facility ID'] = mapping_file['Facility ID'].str.strip()
        return mapping_file
    
    @staticmethod
    def get_facility_details(mapping_file,facility_id):
        facility_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["Facility"].values
        if facility_name.size > 0:
            facility_name = facility_name[0]
            district_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["District"].values
            if district_name.size > 0:
                district_name = district_name[0]
            else:
                district_name = ""
            province_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["Province"].values
            if province_name.size > 0:
                province_name = province_name[0]
            else:
                province_name = ""
        else:
            facility_name = ""
            district_name = ""
            province_name = ""
        return facility_name,district_name,province_name
    
    @staticmethod
    def deduplication(demographics):
        clusters = pandas_dedupe.dedupe_dataframe(demographics,['FirstName','LastName', 'Birthdate', 'Sex', "Street", "City","Education Id"],n_cores = 0)
        return clusters

    @staticmethod
    def cbs_unique_id(record,seq_number):
        if record["person_id"] == "":
            cbs_id = "No Demographics"
        elif record["cluster id"] in cluster_id_dictionary.keys():
            cbs_id = cluster_id_dictionary.get(record["cluster id"])
        else:
            
            first_letter_first_name = str(record["firstname"])[0]
            last_letter_first_name = str(record["firstname"])[-1]
            first_letter_last_name = str(record["Lastname"])[0]
            last_letter_last_name = str(record["Lastname"])[-1]
            first_letter_of_sex = str(record["sex"])[0]
            birthdate = str(record["birthdate"]).replace("/","")
            birthdate = str(record["birthdate"]).replace("-","")
            sequential_number = str(seq_number).rjust(6,"0")
            cbs_id = first_letter_first_name + last_letter_first_name + "-" + first_letter_last_name + last_letter_last_name + "-" + first_letter_of_sex + "-" + birthdate + "-" + sequential_number
            cluster_id_dictionary[ record["cluster id"]] = cbs_id
        return cbs_id

    @staticmethod
    def concat_dataframes(list_of_dataframes):
        df = pd.concat(list_of_dataframes, ignore_index=True, sort=False)
        return df
    
    @staticmethod
    def remove_people_without_positive_result(df,df_positives):
        df = df[df["CBS ID"].isin(list(df_positives["CBS ID"]))]
        return df
    
    @staticmethod
    def get_last_negative_date(df_negatives,df_positives):
        list_all = []
        cbs_ids= list(df_negatives["CBS ID"])
        # Pick the first positive for each cbs id
        df_positives.sort_values(by =['CBS ID','Date of HIV Test'],ascending = True, inplace= True)
        df_positives.drop_duplicates('CBS ID',keep = 'first',inplace= True)
        # Create mapping dictionary
        mapping_dict = dict(zip(df_positives['CBS ID'], df_positives['Date of HIV Test']))
        df_negatives['Date of HIV Test'] = df_negatives["CBS ID"].map(mapping_dict)
        for cbs_id in cbs_ids:
            df_cbs = df_negatives[df_negatives["CBS ID"] == cbs_id]
            sorted_df = df_cbs.sort_values('Last HIV negative date', ascending=False)
            list_negative = [lnd for lnd in sorted_df.iterrows() 
                             if lnd[1]["Last HIV negative date"] + pd.Timedelta(days=1) < lnd[1]["Date of HIV Test"]]
            if len(list_negative)> 0:
                list_all.append(list_negative[0][1])
        list_all_df = pd.DataFrame(list_all)
        list_all_df = list_all_df.drop('Date of HIV Test', axis=1)
        list_all_df = list_all_df[~list_all_df["Event_date"].isna()]
        return list_all_df
    
  

if __name__ == '__main__':

    processing_time = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    extraction  = CBSExtraction("./", "./Jason Files/")
    connection = extraction.get_connection()
    facilities = extraction.get_facility_databases(r"C:\Users\Simba\Desktop\Harare")
    mapping_file = extraction.get_mapping_file()

    #----------------Lists for accumulative data from facilities..........................................
    global_df_demos = pd.DataFrame()
    global_df_hts_positives = pd.DataFrame()
    global_df_hts_negative = pd.DataFrame()
    global_df_cbs = pd.DataFrame()
    global_df_recency = pd.DataFrame()
    global_df_art = pd.DataFrame()
    global_df_art_visit = pd.DataFrame()
    global_df_art_current_status = pd.DataFrame()
    global_df_viral_load = pd.DataFrame()
    global_df_cd4 = pd.DataFrame()
    global_df_tb = pd.DataFrame()
    global_df_tb_screening = pd.DataFrame()
    global_df_transfer_out =  pd.DataFrame()
    global_df_art_who_stage = pd.DataFrame()

    print(">>> Number of databases ", len(facilities))
    

    if Path(extraction.folder_path+ "log.csv").exists():
        os.remove("log.csv")
    if Path(extraction.folder_path+ "demos.csv").exists():
        os.remove("demos.csv")
    
    
    for facility in facilities:
        print(">>>", facility)

        # Restore database
        extraction.restore_database(facility,processing_time)

        # -------------Section1 : Constructing dataframes from database tables -------------------------------------
        print("   >>> Constructing database tables")

        
        # ----- get facility id,version and last time stamp
        df_facility = extraction.extracting_data("""
                            SELECT meta_data FROM mrs.domain_event_entry
                            order by time_stamp desc
                            limit 1
                        """, connection)
        facility_id,version,last_timestamp = extraction.get_facility_identifiers(df_facility.iloc[0][0].decode())

        facility_name , district_name, province_name =  extraction.get_facility_details(mapping_file,facility_id)

        # get facility first time stamp
        first_time_stamp = extraction.extracting_data("""
                                                            SELECT meta_data FROM mrs.domain_event_entry
                                                            order by time_stamp asc
                                                            limit 1
                                                          """, connection)
        get_first_time_stamp = extraction.get_first_time_stamp(first_time_stamp.iloc[0][0].decode())
   

       # Patient ----------------
        patient = extraction.extracting_data("""
                                                SELECT * FROM consultation.patient
                                                          """, connection)
 
        # HTS  ------------
        hts = extraction.extracting_data("""
                                                SELECT * FROM consultation.hts
                                                          """, connection)

        # HTS Screeniing------------
        hts_screening = extraction.extracting_data("""
                                                SELECT * FROM consultation.hts_screening
                                                          """, connection)

        # Person Investigation 
        person_investigation = extraction.extracting_data("""
                                                SELECT * FROM consultation.person_investigation
                                                          """, connection)
  
        # cbs
        cbs = extraction.extracting_data("""
                                                SELECT * FROM consultation.cbs
                                                          """, connection)
        

        # sexual_history ----------------------
        sexual_history = extraction.extracting_data("""
                                                SELECT * FROM consultation.sexual_history
                                                          """, connection)

        # sexual_history_question ----------------------
        sexual_history_question = extraction.extracting_data("""
                                                SELECT * FROM consultation.sexual_history_question
                                                          """, connection)

        # patient_tb_screening ----------------------
        tb_screening = extraction.extracting_data("""
                                                SELECT * FROM consultation.patient_tb_screening
                                                          """, connection)

        # patient_client_profile ----------------------
        patient_client_profile = extraction.extracting_data("""
                                                SELECT * FROM consultation.patient_client_profile
                                                          """, connection)
        
        
         # art_transfer_out ----------------------
        art_transfer_out = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_transfer_out
                                                          """, connection)

         # tb ----------------------
        tb = extraction.extracting_data("""
                                                SELECT * FROM consultation.tb
                                                          """, connection)

        # laboratory_investigation_test ----------------------
        laboratory_investigation_test = extraction.extracting_data("""
                                                SELECT * FROM consultation.laboratory_investigation_test
                                                          """, connection)

        # laboratory_request_order ----------------------
        lab_request = extraction.extracting_data("""
                                                SELECT * FROM consultation.laboratory_request_order
                                                          """, connection)

        # art_visit ----------------------
        art_visit = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_visit
                                                          """, connection)

        # art_who_stage ----------------------
        art_who_stage = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_who_stage
                                                          """, connection)
        
        # art_current_status ----------------------
        art_current_status = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_current_status
                                                          """, connection)
        
        # art_appointment ----------------------
        art_appointment = extraction.extracting_data("""
                                                SELECT * FROM consultation.art_appointment
                                                          """, connection)
        
        # art ----------------------
        art = extraction.extracting_data("""
                                                SELECT * FROM consultation.art
                                                          """, connection)

        # person_diagnosis ----------------------
        person_diagnosis = extraction.extracting_data("""
                                                SELECT * FROM consultation.person_diagnosis
                                                          """, connection)

        # person ----------------------
        demographics = extraction.extracting_data("""
                                                SELECT * FROM client.person
                                                          """, connection)

    
        demographics["birthdate"] = pd.to_datetime(demographics["birthdate"],infer_datetime_format=True,errors='coerce')
        person_diagnosis["date_created"] = pd.to_datetime(person_diagnosis["date_created"],infer_datetime_format=True,errors='coerce')
        patient["time"] = pd.to_datetime(patient["time"],infer_datetime_format=True,errors='coerce')
        hts["time"] = pd.to_datetime(hts["time"],infer_datetime_format=True,errors='coerce')


        # -------------Section2 : Extract data using SQL into ----------------------------------------------------------------------------
        
        print("   >>> Extracting data")

        #---------------a. new HIV positives
        df_hts = duckdb.query("""
                            WITH ranked_messages AS (
                            select 
                            p.person_investigation_id,
                            pt.time as Event_date,
                            p.person_id,
                            h.hts_number as 'HTS Number',
                            reason_for_not_initiating_art,
                            pt.time as 'Date of HIV Test',
                            h.entry_point as 'Entry Point',
                            h.purpose as 'Reason for HIV Test',
                            h.reason_for_not_performing_test as 'Reason for not doing recency test',
                            h.entry_point_id as 'Entry Point Id', h.hts_type as 'Test Method', 
                            h.client_already_positive as 'Already positive',
                            h.client_already_on_art as 'Already on art', p.result as HIV_Result,
                            h.pregnant as 'Pregnant during HIV Test',
                            ROW_NUMBER() OVER (PARTITION BY p.person_id ORDER BY pt.time ASC) as rnt
                            FROM hts h  
                            left join patient pt
                            on  h.patient_id = pt.patient_id
                            left join person_investigation p
                            on h.laboratory_investigation_id = p.person_investigation_id
                            where  p.test = 'HIV' and HIV_Result = 'POSITIVE' and h.purpose <> 'Retesting for ART initiation'
                            )
                            SELECT * FROM ranked_messages WHERE rnt = 1
                        """
                        ).df()
        
        df_hts_from_art = duckdb.query(
                        """
                            select date_of_hiv_test as 'Event_date',
                            person_id, 
                            date_of_hiv_test as 'Date of HIV Test',
                            'POSITIVE' as HIV_Result 
                            from art
                            where person_id not in (select person_id from df_hts)
                        """
                        ).df()

        df_hts_from_cbs = duckdb.query(
                        """
                            select date_of_hiv_test as 'Event_date',
                            person_id, 
                            date_of_hiv_test as 'Date of HIV Test',
                            'POSITIVE' as HIV_Result 
                            from cbs 
                            where person_id not in (select person_id from df_hts) or 
                            person_id not in (select person_id from df_hts_from_art)
                        """
                        ).df()
        
        # Merge df_hts with df_hts_from_art
        df_hts_from_art["Date of HIV Test"] = pd.to_datetime(df_hts_from_art["Date of HIV Test"],errors='coerce').dt.date
        df_hts_from_art["Event_date"] = pd.to_datetime(df_hts_from_art["Event_date"],errors='coerce').dt.date
        df_hts["Date of HIV Test"] = pd.to_datetime(df_hts["Date of HIV Test"],errors='coerce').dt.date
        df_hts["Event_date"] = pd.to_datetime(df_hts["Event_date"],errors='coerce').dt.date
        df_positives_1 = pd.merge(df_hts,df_hts_from_art,how = "outer")

        # Merge df_positives with 
        df_hts_from_cbs["Date of HIV Test"] = pd.to_datetime(df_hts_from_cbs["Date of HIV Test"],errors='coerce').dt.date
        df_hts_from_cbs["Event_date"] = pd.to_datetime(df_hts_from_cbs["Event_date"],errors='coerce').dt.date
        df_positives = pd.merge(df_positives_1,df_hts_from_cbs,how = "outer")

        if df_positives.empty:
                print("    >>> There are no positive cases in the database")
                continue
        
        # ---------------a1. new HIV positives
        df_hts_investigations =  duckdb.query(
                        """
                            WITH ranked_messages AS(
                            select  h.laboratory_investigation_id, h.result,h.time , p.person_investigation_id, p.person_id,
                            ROW_NUMBER() OVER (PARTITION BY p.person_investigation_id ORDER BY time ASC) as rtt
                            from person_investigation p
                            left join laboratory_investigation_test h
                            on p.person_investigation_id = h.laboratory_investigation_id
                            )
                            select * from ranked_messages 
                            where person_investigation_id in 
                            ( select person_investigation_id from df_hts)
                        """
                        ).df()
            
        df_hts_investigations['rtt'] = df_hts_investigations['rtt'].map({1:"HIVtestOneResult", 2: "HIVtestTwoResult",3:"HIVtestThreeResult"})
        df_hts_investigations = df_hts_investigations.pivot(index='person_id', columns=['rtt'], values='result')
        df_hts_investigations = df_hts_investigations.reset_index()

        # ---------------a2. new HIV positives
        df_sexual_history = duckdb.query(
                        """
                            select sexual_history_id,person_id,sexually_active,date from sexual_history
                        """
                        ).df()
        replace_dict = {1:"YES",0:"NO"}
        sexual_history.replace({"sexually_active": replace_dict},inplace=True)
        df_sexual_history= sexual_history[['sexual_history_id', 'person_id',  'sexually_active','date']]


        # ---------------a3. new HIV positives
        df_sexual_history_question = duckdb.query(
                        """
                            select sexual_history_id,question,response_type from sexual_history_question 
                        """
                        ).df()
        df_sexual_history_question.drop_duplicates(subset=["sexual_history_id","question"],inplace = True)
        df_sexual_history_question = df_sexual_history_question.pivot(index='sexual_history_id', columns='question')
        df_sexual_history_question.columns = df_sexual_history_question.columns.droplevel()
        df_sexual_history_question.reset_index(inplace=True)
        df_sexual_history_question = pd.merge(df_sexual_history_question, df_sexual_history , how="left", on=["sexual_history_id"])
        df_sexual_history_question.drop(['sexual_history_id'], axis=1,inplace=True)
        df_sexual_history_question.rename(columns = {'Exchanged sex for  money/material goods':'Exchanged sex for moneymaterial goods',
                                                        'Victim/ Suspected victim of sexual abuse':'Victim Suspected victim of sexual abuse',
                                                        'Unprotected sex without a condom':'Unprotected sex without a condom',
                                                        'sexually_active': 'Sexually Active' }, inplace = True)

        column_order = {'Event_date': 1, 
                            'person_id':2,'Been incarcerated into jail':3,
                    'Exchanged sex for moneymaterial goods':4,
                    'Had Anal Sex':5,'Had sex with male':6,'Had sex with female':7,'Had sex with a sex worker':8,
                    'Victim Suspected victim of sexual abuse':9,'Tattooed with unsterilized instruments':10,
                    'Received medical injections':11, 'Unprotected sex without a condom':12, 'Injected recreational drugs':13,
                    'History of an STI':14, 'Had Oral Sex':15,  'Received blood transfusions':16, 'Sexually Active':17
                            }
        df_sexual_history_question = df_sexual_history_question[[col for col in sorted(df_sexual_history_question.columns, key=lambda x: column_order.get(x, float('inf')))]]
        df_sexual_history_question.drop_duplicates(subset=['person_id'], keep='last',inplace=True)
        
        df_hts_positives_merge = pd.merge(df_positives,df_hts_investigations, on =["person_id"], how ="left")
        df_hts_positives = pd.merge(df_hts_positives_merge,df_sexual_history_question, on =["person_id"] , how = "left")
        df_hts_positives.insert(0,"Facility Id",facility_id)
        df_hts_positives.rename(columns = {"HIV_Result":"Final HIV Result" ,
                                        "reason_for_not_initiating_art":"Reason for not initiating on ART"  },inplace = True)
        
        df_hts_positives.replace(replace_dict, inplace=True)
        df_hts_positives["Date of HIV Test"] = pd.to_datetime(df_hts_positives["Date of HIV Test"],errors='coerce').dt.date
        df_hts_positives = df_hts_positives.sort_values(by=['Final HIV Result'])
        df_hts_positives.drop_duplicates(subset=['Event_date', 'HTS Number','person_id','Date of HIV Test'], keep='last',inplace=True)
        global_df_hts_positives = extraction.concat_dataframes([global_df_hts_positives,df_hts_positives])


        #---------------b. demographics
        df_demographics_extract = duckdb.query(
                        """
                            select 
                            person_id,firstname,lastname as Lastname, birthdate, sex, 
                            self_identified_gender as 'Self Identified Gender',
                            marital_id as 'Marital Id', marital as 'Marital Status',education_id as 'Education Id',
                            education as 'Education',occupation_id as 'Occupation Id' ,occupation as 'Occupation',
                            religion_id as 'Religion id', religion as 'Religion',nationality_id as 'Nationality Id',
                            nationality as Nationality, street as 'Street',city as City, town_id as 'Residential Town Id', 
                            town as 'Residential Town'
                            from demographics
                        """
            ).df()
        
        # df_demographics_extract = df_demographics_extract.sort_values(by=['birthdate'])
        # df_demographics_extract.drop_duplicates(subset=['person_id'], keep='last',inplace = True)
        df_patient_client_profile = duckdb.query(
                        """
                            select 
                            person_id, client_profile as 'Client Profile'
                            from patient_client_profile
                        """
                ).df()
        df_patient_client_profile.drop_duplicates(subset=['person_id'], keep='last',inplace = True)
        df_demographics = pd.merge(df_demographics_extract,df_patient_client_profile,on =["person_id"],how="left")
        df_demographics.insert(0,"Facility Id",facility_id)
        global_df_demos = extraction.concat_dataframes([global_df_demos,df_demographics])


        #---------------c. Extract date of all known hiv negative test from hts
        df_hts_negative = duckdb.query(
                        """
                            select 
                            'hts' as 'Source of Last Negative',
                            pt.time as Event_date,
                            p.person_id,
                            pt.time as 'Last HIV negative date'
                            FROM hts h  
                            left join patient pt
                            on  h.patient_id = pt.patient_id
                            left join person_investigation p
                            on h.laboratory_investigation_id = p.person_investigation_id
                            where  p.test = 'HIV' and p.result in ('Negative' ,'NEGATIVE' ,'NEG','Neg')
                           
                        """
                        ).df()
        
        df_hts_screening_negative = duckdb.query(
                        """
                            select 
                            'HTS Screening' as 'Source of Last Negative',
                            h.date_last_tested as Event_date,
                            p.person_id ,
                            h.date_last_tested as 'Last HIV negative date'
                            from hts_screening h
                            left join patient p
                            on h.patient_id = p.patient_id
                            where result in ('Negative' ,'NEGATIVE' ,'NEG','Neg')
                        """
                        ).df()
        
        df_investigation_negative = duckdb.query(
                        """
                            select 
                            'Investigation' as 'Source of Last Negative',
                            date as Event_date,
                            person_id,
                            date as 'Last HIV negative date'
                            FROM person_investigation 
                            where  test = 'HIV' and result in ('Negative' ,'NEGATIVE' ,'NEG','Neg')  
                            and (person_id not in (select person_id from df_hts_negative)
                            or
                            person_id not in (select person_id from df_hts_screening_negative))
                           
                        """
                        ).df()
        
        

        df_negatives = pd.concat([df_hts_negative,df_investigation_negative,df_hts_screening_negative])
        
        df_negatives["Event_date"] = pd.to_datetime(df_negatives["Event_date"],errors='coerce').dt.date
        df_negatives["Last HIV negative date"] = pd.to_datetime(df_negatives["Last HIV negative date"],errors='coerce').dt.date
        df_negatives.insert(0,"Facility Id",facility_id)
        global_df_hts_negative = extraction.concat_dataframes([global_df_hts_negative,df_negatives])

        #---------------d. cbs
        df_cbs = duckdb.query(
        """
                select date as 'Event_date' ,
                person_id,'Yes' as 'Notified', date as 'Date of Notification' , been_on_prep as 'Been On Prep' 
                from cbs 
        
        """
        ).df()
        df_cbs.insert(0,"Facility Id",facility_id)
        df_cbs = df_cbs[[
            'Event_date', "Facility Id",'Been On Prep',
            "person_id", 'Notified',
            'Date of Notification'
        ]]
        df_cbs["Date of Notification"] = pd.to_datetime(df_cbs["Date of Notification"],errors='coerce').dt.date
        df_cbs["Event_date"] = pd.to_datetime(df_cbs["Event_date"],errors='coerce').dt.date
        df_cbs.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_cbs = extraction.concat_dataframes([global_df_cbs,df_cbs])

        #---------------e. recency
        df_recency = duckdb.query(
                        """
                            select date as 'Event_date',
                            person_id, date as 'Date Recency Done',
                            result as 'Recency Result'
                            from person_investigation 
                            where test = 'HIV-1 Rapid Recency' 
                        """
                        ).df()
        df_recency["Date Recency Done"] = pd.to_datetime(df_recency["Date Recency Done"],errors='coerce').dt.date
        df_recency["Event_date"] = pd.to_datetime(df_recency["Event_date"],errors='coerce').dt.date
        df_recency.insert(0,"Facility Id",facility_id)
        df_recency = df_recency [[
            'Event_date', "Facility Id",
            "person_id",
            'Date Recency Done' ,'Recency Result'
        ]]
        df_recency = df_recency.sort_values(by=['Recency Result'])
        df_recency.drop_duplicates(subset=['Event_date','person_id'], keep='first',inplace = True)
        global_df_recency = extraction.concat_dataframes([global_df_recency,df_recency])

        #---------------f. art
        df_art = duckdb.query(
                    """
                        select  date as 'Event_date',
                        art_id,person_id,art_number as "Art Number",art_cohort_number as "Art Cohort Number",
                        date as "Art_Initiation_Date",date_enrolled as "Date Enrolled into ART"
                        from art 
                    """
                    ).df()
        df_art.insert(0,"Facility Id",facility_id)
        df_art["Art_Initiation_Date"] = pd.to_datetime(df_art["Art_Initiation_Date"],errors='coerce').dt.date
        df_art["Event_date"] = pd.to_datetime(df_art["Event_date"],errors='coerce').dt.date
        df_art = df_art [[
            'Event_date', "Facility Id",
            "person_id",
            "Art Number", "Art Cohort Number", "Art_Initiation_Date","Date Enrolled into ART"
        ]]
        df_art = df_art.sort_values(by=['Art Number'])
        df_art.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_art = extraction.concat_dataframes([global_df_art,df_art])

        #---------------g. art_visit
        df_art_visit = duckdb.query(
                """
                    select p.time as 'Event_date', 
                    p.time as "Art visit date", p.person_id, a.visit_type as 'Art visit type',
                    a.tb_status as 'TB Status', a.family_planning_status, a.lactating_status
                    from art_visit a left join patient p
                    on a.patient_id = p.patient_id
                    
                """
                ).df() 
        df_art_visit.insert(0,"Facility Id",facility_id)
        df_art_visit["Art visit date"] = pd.to_datetime(df_art_visit["Art visit date"],errors='coerce').dt.date
        df_art_visit["Event_date"] = pd.to_datetime(df_art_visit["Event_date"],errors='coerce').dt.date
        df_art_visit = df_art_visit[[
                'Event_date', "Facility Id",
                'person_id',
                "Art visit date",'Art visit type',
                'TB Status'
            ]]
        df_art_visit = df_art_visit.sort_values(by=['Art visit type'])
        df_art_visit.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        df_art_visit = df_art_visit.replace('Presumptive-if there are signs', 'Suspect or Presumptive')
        df_art_visit = df_art_visit.replace('Screened and has no signs', 'Screened with no signs')
        df_art_visit = df_art_visit.replace('Tb status not assesssed', '')
        global_df_art_visit = extraction.concat_dataframes([global_df_art_visit,df_art_visit])

        #---------------h. art current status
        df_art_current_status = duckdb.query(
                """
                    select a.date as 'Event_date', 
                    a.date as 'Art Current Status Date',a.art_id ,a.regimen as regimen , a.state as 'ARV Status', a.art_initiation_category as 'Art Initiation Category',
                    art.person_id,a.regimen_id as 'Regimen Id'
                    from art_current_status a 
                    left join art 
                    on a.art_id = art.art_id
                    
                """
                ).df()
        df_art_current_status.insert(0,"Facility Id",facility_id)
        df_art_current_status["Art Current Status Date"] = pd.to_datetime(df_art_current_status["Art Current Status Date"],errors='coerce').dt.date
        df_art_current_status["Event_date"] = pd.to_datetime(df_art_current_status["Event_date"],errors='coerce').dt.date
        df_art_current_status = df_art_current_status[[
        'Event_date',"Facility Id",'person_id', 'ARV Status','Regimen Id','regimen','Art Initiation Category'
        ]]
        df_art_current_status = df_art_current_status.sort_values(by=['regimen'])
        df_art_current_status.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_art_current_status = extraction.concat_dataframes([global_df_art_current_status,df_art_current_status])

        #---------------i. art who stage
        df_art_who_stage = duckdb.query(
                """
                    select  a.date as 'Event_date',
                    p.person_id , a.art_id, a.date as 'Who Stage Date',
                    a.date as 'Outcome Date',
                    a.stage as 'Who Stage', a.follow_up_status as 'Art Outcome'
                    from art_who_stage a left join art p
                    on a.art_id  = p.art_id
                    
                """
            ).df()
        df_art_who_stage.insert(0,"Facility Id",facility_id)
        df_art_who_stage["Who Stage Date"] = pd.to_datetime(df_art_who_stage["Who Stage Date"],errors='coerce').dt.date
        df_art_who_stage["Event_date"] = pd.to_datetime(df_art_who_stage["Event_date"],errors='coerce').dt.date
        df_art_who_stage = df_art_who_stage [[
            'Event_date', "Facility Id",
            'person_id','Who Stage Date','Who Stage',
            'Art Outcome'
        ]]
        df_art_who_stage = df_art_who_stage.sort_values(by=['Who Stage'])
        df_art_who_stage.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_art_who_stage = extraction.concat_dataframes([global_df_art_who_stage,df_art_who_stage])

        #---------------j. viral load
        df_viral_load = duckdb.query(
                """
                    select date as "Event_date",
                    person_id,date as "Date at which Viral Load result was issued",
                    date as "Date for which Viral Load was taken",
                    'TRUE' as "Viral Load Sample submitted to lab",
                    'TRUE' as "Was Viral Load result issued",
                    result as "Viral Load result",
                    from person_investigation
                    
                """
            ).df()
        df_viral_load.insert(0,"Facility Id",facility_id)
        df_viral_load["Date at which Viral Load result was issued"] = pd.to_datetime(df_viral_load["Date at which Viral Load result was issued"],errors='coerce').dt.date
        df_viral_load["Event_date"] = pd.to_datetime(df_viral_load["Event_date"],errors='coerce').dt.date
        df_viral_load = df_viral_load[[
                    "Event_date","Facility Id",
                    'person_id',
                    "Date at which Viral Load result was issued",
                    "Date for which Viral Load was taken",
                    "Viral Load Sample submitted to lab",
                    "Was Viral Load result issued",
                    "Viral Load result"
        ]]
        df_viral_load = df_viral_load.sort_values(by=['Viral Load result'])
        df_viral_load.drop_duplicates(subset=['Event_date','person_id'], keep='last', inplace = True)
        global_df_viral_load = extraction.concat_dataframes([global_df_viral_load,df_viral_load])

        #---------------k. cd4
        df_cd4 = duckdb.query(
                """
                    select date as "Event_date",
                    person_id, date as 'Date at which cd4 sample was taken',
                    date as 'Date at which cd4 result was issued',
                    'TRUE' as "CD4 Sample submitted to lab",
                    'TRUE' as "Was cd4 result issued",
                    result as 'CD4 Count'
                    from person_investigation
                  
                """
                ).df()
        df_cd4.insert(0,"Facility Id",facility_id)
        df_cd4["Date at which cd4 result was issued"] = pd.to_datetime(df_cd4["Date at which cd4 result was issued"],errors='coerce').dt.date
        df_cd4["Event_date"] = pd.to_datetime(df_cd4["Event_date"],errors='coerce').dt.date
        df_cd4 = df_cd4 [[
            'Event_date',"Facility Id",
            'person_id',
            'Date at which cd4 sample was taken',
            'Date at which cd4 result was issued',
            "CD4 Sample submitted to lab",
            "Was cd4 result issued",
            'CD4 Count',
        ]]
        df_cd4 = df_cd4.sort_values(by=['CD4 Count'])
        df_cd4.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        global_df_cd4 = extraction.concat_dataframes([global_df_cd4,df_cd4])

        #---------------l. TB
        df_TB = duckdb.query(
                """
                    select date as 'Event_date',
                    person_id,date as 'TB Treatment Start Date',type_of_tb as 'Type Of TB',
                    tb_disease_site as 'TB Disease Site',
                    tb_disease_type as 'TB Disease Type',outcome as 'TB Outcome'
                    from tb
                    
                """
            ).df()
        df_TB.insert(0,"Facility Id",facility_id)
        df_TB["TB Treatment Start Date"] = pd.to_datetime(df_TB["TB Treatment Start Date"],errors='coerce').dt.date
        df_TB["Event_date"] = pd.to_datetime(df_TB["Event_date"],errors='coerce').dt.date
        df_TB = df_TB[[
                'Event_date',"Facility Id",
                'person_id','TB Treatment Start Date',
                'TB Outcome','Type Of TB','TB Disease Site','TB Disease Type'
            ]]
        df_TB = df_TB.sort_values(by=['TB Outcome'])
        df_TB.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
        global_df_tb = extraction.concat_dataframes([global_df_tb,df_TB])

        #---------------m. TB screening
        df_TB_Screening = duckdb.query(
                """
                    select t.time as 'Event_date',
                    t.person_id , p.presumptive as 'TB Screened', 
                    from tb_screening p
                    left join patient t 
                    on p.patient_id = t.patient_id
                    
                """
                ).df()
        df_TB_Screening.insert(0,"Facility Id",facility_id)
        df_TB_Screening = df_TB_Screening.replace('\x01', 'Suspect or Presumptive')
        df_TB_Screening = df_TB_Screening.replace('\\0', 'Screened with no signs')
        df_TB_Screening = df_TB_Screening[[
            'Event_date', "Facility Id",
            'person_id',
            'TB Screened'
        ]]
        df_TB_Screening["Event_date"] = pd.to_datetime(df_TB_Screening["Event_date"],errors='coerce').dt.date
        df_TB_Screening.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
        global_df_tb_screening = extraction.concat_dataframes([global_df_tb_screening,df_TB_Screening])

        #---------------n. transfer out
        df_transfer_out = duckdb.query(
                    """
                    select 
                    a.person_id , t.art_id , t.transfer_out_date as 'Event_date',
                    t.transfer_out_date as 'Transfer Out date',
                    t.transfer_reason as 'Transfer Reason', t.transfer_facility_id
                    from art_transfer_out t left join art a
                    on t.art_id = a.art_id
                    where a.person_id in (select person_id from df_hts_positives)
                """
                ).df()
        df_transfer_out["Event_date"] = pd.to_datetime(df_transfer_out["Event_date"],errors='coerce').dt.date
        df_transfer_out = df_transfer_out.sort_values(by=['transfer_facility_id'])
        df_transfer_out.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
        df_transfer_out.insert(0,"Facility Id",facility_id)
        global_df_transfer_out = extraction.concat_dataframes([global_df_transfer_out,df_transfer_out])

    # -------------Section2 -------------------------------------- -------------------------------------
    #---------------a. deduplication
    df_demographics_for_dedup = global_df_demos[["Facility Id",
            "person_id", "firstname", "Lastname", "birthdate", "sex",
            'Self Identified Gender', 'Street',
            "City", 'Education Id', 'Occupation Id','Religion id'
            ]]
    
    global_df_demos.to_csv("Demographics original.csv")
    
    df_demographics_for_dedup["birthdate"] = pd.to_datetime(df_demographics_for_dedup["birthdate"],errors='coerce').dt.date
    df_demographics_for_dedup = df_demographics_for_dedup.sort_values(by=['birthdate'])
    df_demographics_for_dedup.drop_duplicates(subset=['person_id'], keep='last',inplace=  True)
    df_demographics_for_dedup.fillna("NULL",inplace=True)
    df_demographics_for_dedup.to_csv("demographics.csv")
    input("enter")
    df_for_dedup = pd.read_csv("demographics.csv")
    clusters = extraction.deduplication(df_for_dedup)

    print("    >>> Generating CBS ID")
    #---------------b. Generate CBS ID 
    cluster_id_dictionary = {}
    seq_number = 0
    
    
    cbs_ids =[]
    for i,row in clusters.iterrows():
        if row["cluster id"] in cluster_id_dictionary.keys() or row["firstname"] == "No Name":
            seq_number = seq_number
        else:
            seq_number = seq_number + 1
            extraction.cbs_unique_id(row,seq_number)
        cbs_ids.append(extraction.cbs_unique_id(row,seq_number))
    clusters["CBS ID"] = cbs_ids
    clusters.sort_values(by = ["cluster id"],inplace = True)

    person_id_cbs_id = clusters [[
        "person_id","CBS ID"
    ]]
    dictionary_mapping = person_id_cbs_id.set_index('person_id')['CBS ID'].to_dict()
  
    print("    >>> Add CBS ID to all dataframes")
    #-------------- c. Add CBS ID to every global dataframe
    list_of_globals = [global_df_demos, global_df_hts_positives,global_df_hts_negative,
              global_df_cbs, global_df_recency, global_df_art,
              global_df_art_visit, global_df_art_current_status, global_df_viral_load,
              global_df_cd4,global_df_art_who_stage,global_df_tb,global_df_tb_screening,
              global_df_transfer_out
              ]
    for df in list_of_globals:
        df['CBS ID'] = df['person_id'].map(dictionary_mapping)
    
    #----------------c2 Add demographics to dataset with positives
    global_df_demos.drop(['Facility Id'], axis=1, inplace=True)
    merge_demos = pd.merge(global_df_hts_positives,global_df_demos, on =["person_id","CBS ID"] , how = "left")
    merge_demos.sort_values(["CBS ID"],inplace=True)

    #---------------d. Merging files on CBS ID
    # --------------i. Merge with negatives....................
    # Remove people who do not have positive test or art initiation
    print("    >>> Merging dataframes")
    global_negatives = extraction.remove_people_without_positive_result(global_df_hts_negative,global_df_hts_positives)
    global_negatives_sorted = global_negatives.sort_values(by=['Source of Last Negative']).drop_duplicates(['person_id','Last HIV negative date'], keep='first')

    merge_demos["Event_date"] = pd.to_datetime(merge_demos["Event_date"],errors='coerce').dt.date

    merge_negatives1  = extraction.get_last_negative_date(global_negatives_sorted, merge_demos)
    merge_negatives1.drop(['person_id'], axis=1, inplace=True)
    merge_negatives2 = pd.merge(merge_demos, merge_negatives1, on =["Event_date","CBS ID","Facility Id"] , how = "outer")

    merge_negatives2.sort_values(["CBS ID","Event_date"],inplace=True)
    
    #---------------ii. Merge with recency....................
    global_recency = extraction.remove_people_without_positive_result(global_df_recency,global_df_hts_positives)
    global_recency = global_recency.sort_values(by=['Event_date']).drop_duplicates(['person_id'], keep='first')
    global_recency["Event_date"] = pd.to_datetime(global_recency["Event_date"],errors='coerce').dt.date
    global_recency.drop(['person_id'], axis=1, inplace=True)
    merge_recency = pd.merge(merge_negatives2, global_recency, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_recency.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------iii Merge with art data..................
    global_art = extraction.remove_people_without_positive_result(global_df_art,global_df_hts_positives)
    global_art = global_art.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_art["Event_date"] = pd.to_datetime(global_art["Event_date"],errors='coerce').dt.date
    global_art.drop(['person_id'], axis=1, inplace=True)
    merge_art = pd.merge(merge_recency,global_art, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_art.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------iv Merge with art visit..................
    global_art_visit = extraction.remove_people_without_positive_result(global_df_art_visit,global_df_hts_positives)
    global_art_visit = global_art_visit.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_art_visit["Event_date"] = pd.to_datetime(global_art_visit["Event_date"],errors='coerce').dt.date
    global_art_visit.drop(['person_id'], axis=1, inplace=True)
    merge_art_visit = pd.merge(merge_art,global_art_visit, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_art_visit.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------v Merge with art current..................
    global_art_current_status = extraction.remove_people_without_positive_result(global_df_art_current_status,global_df_hts_positives)
    global_art_current_status = global_art_current_status.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_art_current_status["Event_date"] = pd.to_datetime(global_art_current_status["Event_date"],errors='coerce').dt.date
    global_art_current_status.drop(['person_id'], axis=1, inplace=True)
    merge_art_current = pd.merge(merge_art_visit,global_art_current_status, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_art_current.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------vi Merge with who stage..................
    global_art_who_stage = extraction.remove_people_without_positive_result(global_df_art_who_stage,global_df_hts_positives)
    global_art_who_stage = global_art_who_stage.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_art_who_stage["Event_date"] = pd.to_datetime(global_art_who_stage["Event_date"],errors='coerce').dt.date
    global_art_who_stage.drop(['person_id'], axis=1, inplace=True)
    merge_who = pd.merge(merge_art_current,global_art_who_stage, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_who.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------vii Merge with viral load..................
    global_viral_load = extraction.remove_people_without_positive_result(global_df_viral_load,global_df_hts_positives)
    global_viral_load = global_viral_load.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_viral_load["Event_date"] = pd.to_datetime(global_viral_load["Event_date"],errors='coerce').dt.date
    global_viral_load.drop(['person_id'], axis=1, inplace=True)
    merge_vl = pd.merge(merge_who,global_viral_load, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_vl.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------viii Merge with cd4..................
    global_cd4 = extraction.remove_people_without_positive_result(global_df_cd4,global_df_hts_positives)
    global_cd4 = global_cd4.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_cd4["Event_date"] = pd.to_datetime(global_cd4["Event_date"],errors='coerce').dt.date
    global_cd4.drop(['person_id'], axis=1, inplace=True)
    merge_cd4 = pd.merge(merge_vl,global_cd4, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_cd4.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------ix Merge with tb..................
    global_tb = extraction.remove_people_without_positive_result(global_df_tb,global_df_hts_positives)
    global_tb = global_tb.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_tb["Event_date"] = pd.to_datetime(global_tb["Event_date"],errors='coerce').dt.date
    global_tb.drop(['person_id'], axis=1, inplace=True)
    merge_tb = pd.merge(merge_cd4,global_tb, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_tb.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------x Merge with tb screening..................
    global_tb_screening = extraction.remove_people_without_positive_result(global_df_tb_screening,global_df_hts_positives)
    global_tb_screening = global_tb_screening.sort_values(by=['Event_date']).drop_duplicates( keep  = 'first')
    global_tb_screening["Event_date"] = pd.to_datetime(global_tb_screening["Event_date"],errors='coerce').dt.date
    global_tb_screening.drop(['person_id'], axis=1, inplace=True)
    merge_tb_screening = pd.merge(merge_tb,global_tb_screening, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_tb_screening.sort_values(["CBS ID","Event_date"],inplace=True)

     #----------------xi Merge with tb screening..................
    global_transfer_out = extraction.remove_people_without_positive_result(global_df_transfer_out,global_df_hts_positives)
    global_transfer_out = global_transfer_out.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_transfer_out["Event_date"] = pd.to_datetime(global_transfer_out["Event_date"],errors='coerce').dt.date
    global_transfer_out.drop(['person_id'], axis=1, inplace=True)
    merge_transfer_out = pd.merge(merge_tb_screening,global_transfer_out, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_transfer_out.sort_values(["CBS ID","Event_date"],inplace=True)

    #----------------xii Merge with cbs..................
    global_cbs = extraction.remove_people_without_positive_result(global_df_cbs,global_df_hts_positives)
    global_cbs = global_df_cbs.sort_values(by=['Event_date']).drop_duplicates(keep  = 'first')
    global_cbs["Event_date"] = pd.to_datetime(global_cbs["Event_date"],errors='coerce').dt.date
    global_cbs.drop(['person_id'], axis=1, inplace=True)
    merge_cbs = pd.merge(merge_transfer_out,global_cbs, on =["Event_date","CBS ID","Facility Id"] , how = "outer")
    merge_cbs.sort_values(["CBS ID","Event_date"],inplace=True)
    merge_cbs.rename(columns = {'Event_date':'Event date'}, inplace = True)

    merge_cbs = merge_cbs [['person_id',
            'CBS ID','Event date', 'Facility Id',
            'Client Profile', 'birthdate', 'sex', 'Self Identified Gender', 
            'Marital Status',  'Education','Occupation', 'Religion', 
            'Nationality', 'Residential Town', 'Date of HIV Test', 
            'Reason for HIV Test',
            'Entry Point', 'Pregnant during HIV Test',
            'Already positive', 'Already on art', 'Test Method', 
            'Source of Last Negative','Last HIV negative date',
            'HIVtestOneResult','HIVtestTwoResult', 'HIVtestThreeResult','Final HIV Result',
            'Been incarcerated into jail',
            'Exchanged sex for moneymaterial goods', 'Had Anal Sex',
            'Had sex with male', 'Had sex with female', 'Had sex with a sex worker',
            'Victim Suspected victim of sexual abuse',
            'Tattooed with unsterilized instruments', 'Received medical injections',
             'Injected recreational drugs',
            'History of an STI', 'Had Oral Sex', 'Received blood transfusions',
            'Sexually Active', 'Notified','Date of Notification', 'Been On Prep',
            'Date Recency Done', 'Recency Result','Reason for not doing recency test',
            'Art Number',
            'Art Cohort Number', 'Art_Initiation_Date', 'Date Enrolled into ART','Reason for not initiating on ART',
            'Art visit date', 'Art visit type',
            'ARV Status', 'regimen',
            'Art Initiation Category','Who Stage Date', 'Who Stage', 
            'Date for which Viral Load was taken',
            'Viral Load Sample submitted to lab',
            'Was Viral Load result issued',
            'Date at which Viral Load result was issued',
            'Viral Load result',
            'Date at which cd4 sample was taken',
            'Date at which cd4 result was issued', 'CD4 Sample submitted to lab',
            'Was cd4 result issued',
            'CD4 Count','TB Treatment Start Date',
            'TB Outcome', 'Type Of TB', 'TB Disease Site', 'TB Disease Type',
            'Art Outcome',
            'Transfer Out date',
            'Transfer Reason',
            'transfer_facility_id'
    ]]
    
    filename = datetime.today().strftime('%d%m%Y')
    merge_cbs.to_csv("CBS_" + filename + ".csv",index=False)

>>> Number of databases  3
>>> C:\Users\Simba\Desktop\Harare\Avondale30june23.sql
DROPPED [client]


DROPPED [consultation]
DROPPED [deduplication]
DROPPED [facility]
DROPPED [mrs]
>>> Restoring database
DATABASE RESTORE [SUCCESSFUL>>>]
   >>> Constructing database tables
   >>> Extracting data
>>> C:\Users\Simba\Desktop\Harare\belvedere010723.sql
DROPPED [client]
DROPPED [consultation]
DROPPED [deduplication]
DROPPED [facility]
DROPPED [mrs]
DROPPED [provider]
DROPPED [report]
DROPPED [terminology]
DROPPED [zimepms]
>>> Restoring database
DATABASE RESTORE [SUCCESSFUL>>>]
   >>> Constructing database tables
   >>> Extracting data
>>> C:\Users\Simba\Desktop\Harare\BreasideJul12.sql
DROPPED [client]
DROPPED [consultation]
DROPPED [deduplication]
DROPPED [facility]
DROPPED [mrs]
DROPPED [provider]
DROPPED [report]
DROPPED [terminology]
DROPPED [zimepms]
>>> Restoring database
   >>> Constructing database tables
   >>> Extracting data
Importing data ...
Reading from dedupe_dataframe_learned_settings
Clustering...
# duplicate sets 31589
    >>> Generating CBS ID
    >>> Add CBS ID to all d